# Sovereign-Mahjong V6 — Kaggle End-to-End Pipeline (Convert + Phase 0 + Phase 1/2 RL + Phase 3 SD-TAP + Strict Eval)

本 Notebook 专为 **Kaggle GPU（T4 / P100 / L4 / H100）** 优化，完整覆盖 **Sovereign-Mahjong V6** 从 2026 天凤凤桌牌谱下载、GPU 无损批转换，到 Phase 0-A/B/C、Phase 1/2 PureJaxRL、Phase 3 SD-TAP 行为蒸馏与严格 Duplicate 1v3 评测的全流程。

### 核心数学硬门控（全流程强制校验）
1. **数据零损失**：CPU 精确 136 牌山重建 + GPU `vmap(lax.scan)` 批重放 `mahjax` 规则引擎（自动补全隐式 `PASS` 与 6 类流局，`float16` 无损存储 `{0, 1, k/4, k/8}` 空间特征，约 `235 B/步`）。
2. **Phase 0-B (`GRP_base`)**：10 步 `log_sinkhorn_4x4` Birkhoff 误差 `< 1e-4`、凤桌 7d 精确零和误差 `< 1e-4`、留出集 `ECE < 0.04`；训后严格冻结作为 DAG 防火墙。
3. **Phase 0-A (`BeliefNet` + `T_cal`)**：36× 等变（$S_3^{	ext{suit}} 	imes S_3^{	ext{sangen}}$）+ 严格**无加权**先验偏置 BCE（`-4.184`）+ 独立留出集温度校准 `T_cal`；硬门控：`ROC-AUC >= 0.85`、`mean(p_cal)/mean(y_ron) ≈ 1.00`。
4. **Phase 0-C（严格同构 `SovereignPolicyNet` BC）**：冻结 `B_EMA(T_cal) + GRP_base` 注入解析物理三联体，`beta_risk / beta_offense` 豁免权重衰减端到端学习。
5. **Phase 1/2 (`train_rl_v6`) & Phase 3 (`distill_sdtap`)**：冻策略预热 + Per-Rollout 单快照零重编译 PFSP + 双速率 Simplex TD($\lambda$) + 无偏 Running-RMS（不减均值）+ 外循环重置仅清零 AdamW `m_t` 完整保留 `v_t` + 三重护甲安全盾 + 自愈熔断 + 行为空间自蒸馏。
6. **严格评测标准**：固定牌山全半庄 Duplicate 1v3（`round_mode="half"`, `round_end="confirm"`，4 席轮换配对 95% 置信区间，天凤凤桌 7d `[+90, +45, 0, -135]`）+ 零样本多规则切换（`7d / 10d / M-League`）。

7. **分版本存档与实时基线对抗（防盲训）**：RL 期间每 `RL_VERSION_EVERY` 步保存轻量独立版本 `rl_upd_XXXX.pkl`（约 6.7 MB），同时用**零重编译** `CompiledDuplicate1v3Evaluator` 在固定验证牌山（`RL_EVAL_SEED=2026`）上对战 `3× phase0c_bc_7d`、记录 `rl_eval_history.json` 并自动维护最优版本 `rl_best.pkl`；最终评测切换至完全独立的 `EVAL_TEST_SEED=9999` 防止验证集过拟合。

> **Kaggle 磁盘提示**：原始 451 MB Parquet 放在 `/kaggle/tmp/`（不占 20 GB 持久化输出配额）；转换后的分片与所有模型权重/报表持久化保存在 `/kaggle/working/`。

> **性能修复分支**：本 Notebook 默认拉取 `perf/jax-throughput-kaggle`（合并回 `main` 后改回 `main`）。
> 体检结论、实测数字与 Kaggle T4×2 指南见 `docs/performance.md`；**2.5 节**可直接跑 `num_envs` batch sweep。


## 1. 环境检查、拉取仓库与依赖安装

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"

# 1. Inspect GPU
!nvidia-smi

# 2. Clone repository (or reuse existing /kaggle/working/mahjax)
REPO_URL = "https://github.com/myouo/mahjax.git"
# Performance fixes live on this branch. After merging it into main, set this back to "main"
# (or export MAHJAX_REPO_BRANCH=main before running the notebook).
REPO_BRANCH = os.environ.get("MAHJAX_REPO_BRANCH", "perf/jax-throughput-kaggle")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TMP_ROOT = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else (WORK_ROOT / "tmp")
TMP_ROOT.mkdir(parents=True, exist_ok=True)

# Persistent XLA compile cache. Kaggle's ~ is not preserved across sessions, so keep it
# under /kaggle/working and save it as an Output/Dataset if you want to reuse it next session.
JAX_CACHE_DIR = WORK_ROOT / ".jax_cache"
JAX_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(JAX_CACHE_DIR))

REPO_DIR = WORK_ROOT / "mahjax"
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=False)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"],
        check=False,
    )

_active_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip()
_head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True
).strip()
print(f"[Repo] branch={_active_branch} HEAD={_head}")
if _active_branch != REPO_BRANCH:
    print(
        f"WARNING: expected branch {REPO_BRANCH!r} but got {_active_branch!r}. "
        "Push the perf branch, or set MAHJAX_REPO_BRANCH=main after merging."
    )

# 3. Install required packages (keep Kaggle's preinstalled CUDA JAX if already working)
def _ensure_pkg(import_name: str, pip_spec: str) -> None:
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec], check=True)

_ensure_pkg("jax", "jax[cuda12]")
_ensure_pkg("flax", "flax>=0.10.0")
_ensure_pkg("optax", "optax>=0.2.4")
_ensure_pkg("pyarrow", "pyarrow")
_ensure_pkg("huggingface_hub", "huggingface_hub")

# Editable install of mahjax + mahjong_rl
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# Evict any stale mahjong_rl modules from Jupyter sys.modules after git pull
for _mod in [m for m in list(sys.modules) if m == "mahjong_rl" or m.startswith("mahjong_rl.")]:
    del sys.modules[_mod]

import importlib.util

_has_perf = importlib.util.find_spec("mahjong_rl.jax_runtime") is not None
print("[Perf] mahjong_rl.jax_runtime available:", _has_perf)
if not _has_perf:
    print("WARNING: performance fixes are missing; check out the perf branch before the sweep.")

import jax
try:
    from mahjong_rl.jax_runtime import enable_persistent_compilation_cache
    print("JAX compile cache:", enable_persistent_compilation_cache(str(JAX_CACHE_DIR)))
except Exception as _exc:
    print("WARN: persistent compile cache unavailable:", _exc)
print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())
assert any(d.platform == "gpu" or d.platform == "tpu" for d in jax.devices()), (
    "WARNING: No GPU/TPU accelerator detected! Enable GPU (T4 / P100) in Kaggle Notebook Settings."
)

## 2. 全局超参、路径与 Kaggle 预算配置

In [ ]:
import subprocess
from pathlib import Path
import jax

# ==============================================================================
# Auto-Detect Multi-GPU Count & Per-GPU VRAM (saturates Kaggle T4x2 / P100 / L4 / H100!)
# ==============================================================================
N_DEVICES = jax.local_device_count()
try:
    vram_lines = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()
    VRAM_MB_PER_GPU = int(min(int(x.strip()) for x in vram_lines if x.strip()))
except Exception:
    VRAM_MB_PER_GPU = 15000

VRAM_GB = VRAM_MB_PER_GPU / 1024.0

if VRAM_GB >= 22.0:
    _PER_DEV_CONV_B, _PER_DEV_BEL_B, _PER_DEV_BC_B, _PER_DEV_RL_ENVS, _PER_DEV_RL_MB, _EVAL_CHUNK = 384, 1536, 768, 48, 1536, 16
elif VRAM_GB >= 14.0:
    # Kaggle T4 (15 GB x 2) / P100 (16 GB) preset. 2026-11 sweep: 64 envs/GPU (128 global)
    # gave 1,766 steady sps vs 1,174 at 16/GPU and 1,486 at 32/GPU. Re-sweep 128 vs 256 in 2.5.
    _PER_DEV_CONV_B, _PER_DEV_BEL_B, _PER_DEV_BC_B, _PER_DEV_RL_ENVS, _PER_DEV_RL_MB, _EVAL_CHUNK = 256, 1024, 512, 64, 1024, 16
else:
    # 6-12 GB single GPU fallback
    _PER_DEV_CONV_B, _PER_DEV_BEL_B, _PER_DEV_BC_B, _PER_DEV_RL_ENVS, _PER_DEV_RL_MB, _EVAL_CHUNK = 96, 512, 512, 16, 512, 8

CFG = {
    # Stage switches (existing checkpoints in CKPT_DIR / PREEXISTING_CKPT_DIR are auto-reused!)
    "RUN_DOWNLOAD": False,                     # Set True only if converting raw parquets from scratch
    "RUN_CONVERT": False,                      # Set True only if converting raw parquets from scratch
    "RUN_PHASE0": True,
    "RUN_PHASE12_RL": True,
    "RUN_PHASE3_SDTAP": True,
    "RUN_DUPLICATE_EVAL": True,

    # Fine-grained force-retrain switches (False = auto-skip if .pkl exists; True = retrain & overwrite)
    "FORCE_RETRAIN_GRP": False,
    "FORCE_RETRAIN_BELIEF": False,
    "FORCE_RETRAIN_BC": False,
    "FORCE_RETRAIN_RL": False,                  # Upgrading to symmetric delta_u_head Oracle requires True on 1st clean run (Phase 0 grp/belief/bc stay reused!)

    # Multi-GPU Pipelined Conversion (pmap across ALL GPUs + GPU float16 + zlib L3 + CPU prefetch!)
    # On Kaggle T4x2: 512 kyokus in parallel -> ~4-6x faster than single-GPU default!
    "START_GAME": 0,
    "MAX_GAMES": 10000,
    "FORCE_RECONVERT": False,                   # True -> delete existing shards and reconvert from scratch
    "ALLOW_PARTIAL_SHARDS": False,              # False -> auto-RESUME incomplete shards until MAX_GAMES is reached
    "ROUND_BATCH_SIZE": _PER_DEV_CONV_B * N_DEVICES,
    "MAX_STEPS_PER_ROUND": 144,                 # Covers >99.98% of Phoenix kyokus (25% less padding!)
    "SHARD_SIZE": 50000,
    "RON_K_MAX": _PER_DEV_CONV_B * 80,          # Per-GPU static compaction budget (0 overflows)
    "COMPRESS_LEVEL": 3,                        # Fast compact zlib level 3 (~3x faster, ~14MB/shard)

    # Optional: reuse pre-converted shards/checkpoints from a mounted Kaggle Dataset
    "PREEXISTING_SHARD_DIR": "",
    "PREEXISTING_CKPT_DIR": "",

    # Phase 0 offline supervised hyperparameters (auto-scaled across ALL GPUs via pmap!)
    "VAL_SHARDS": 4,
    "GRP_EPOCHS": 10,
    "GRP_BATCH_SIZE": 2048,
    "BELIEF_EPOCHS": 5,
    "BELIEF_BATCH_SIZE": _PER_DEV_BEL_B * N_DEVICES,
    "BC_EPOCHS": 5,
    "BC_BATCH_SIZE": _PER_DEV_BC_B * N_DEVICES,

    # Phase 1/2 Multi-GPU PureJaxRL (pmap rollouts + synchronized pmean grads across ALL GPUs!)
    "RL_NUM_ENVS": _PER_DEV_RL_ENVS * N_DEVICES,  # global batch = per-device * N_DEVICES
    "RL_NUM_STEPS": 256,
    # Batch sweep (Section 2.5). 32/64/128 already measured; now check 128 vs 256.
    # Set PERF_BENCH_BATCHES=[] to skip.
    "PERF_BENCH_BATCHES": [128, 256],
    "PERF_BENCH_UPDATES": 4,
    "RL_WARMUP_UPDATES": 80,
    # NOTE: one update consumes RL_NUM_ENVS * RL_NUM_STEPS transitions. If you change
    # RL_NUM_ENVS (e.g. 64 -> 128), rescale this and the update-based schedules below
    # (WARMUP / PFSP / EVAL) if you want to keep the same total environment steps.
    "RL_NUM_UPDATES": 1000,                    # Target cumulative updates (resumes automatically from last ckpt!)
    "RL_OUTER_RESET_STEPS": 0,                 # 0 = permanent BC anchor (prevents drifted outer-reset ratcheting)
    "RL_PFSP_SNAPSHOT_INTERVAL": 200,          # Healthy PFSP snapshot interval when RL_OUTER_RESET_STEPS=0
    "RL_MINIBATCH_SIZE": _PER_DEV_RL_MB * N_DEVICES,
    "RL_UPDATE_EPOCHS": 2,
    "RL_POLICY_LR": 3e-5,
    "RL_LAMBDA_INTRA": 1.0,
    "RL_LAMBDA_INTER": 0.0,
    "RL_MASK_UNTERMINATED_TAIL": True,
    "RL_FREEZE_BELIEF": True,
    "RL_ALPHA_RNAD": 0.12,
    "RL_RHO_MIX_BC": 0.10,
    "RL_RHO_MIX_UNIFORM": 0.005,
    "RL_ENTROPY_COEF": 0.002,
    "RL_PHOENIX_7D_ONLY": True,
    "RL_PFSP_FRAC": 0.625,
    "RL_OPP_TEMPERATURE": 0.35,
    "RL_ORACLE_REPLAY_UPDATES": 6,
    "RL_RESUME": True,                          # True -> auto-resume from rl_resume_state.pkl / rl_phase2.pkl up to RL_NUM_UPDATES
    "RL_CHECKPOINT_EVERY": 50,                  # Atomic resumable checkpoint every N updates (+ auto-upload to HF if configured)
    # Versioned checkpoints + periodic zero-recompile Duplicate 1v3 eval vs frozen BC
    "RL_VERSION_EVERY": 250,                   # Save standalone lightweight rl_upd_XXXX.pkl (~6.7 MB) every N updates (0=disable)
    "RL_EVAL_EVERY": 250,                      # Run compiled Duplicate 1v3 eval vs 3x frozen BC every N updates (0=disable; multiple of RL_CHECKPOINT_EVERY)
    "RL_EVAL_WALLS": 128,                      # Fixed paired validation walls per periodic eval (128 walls = 512 half-games, CI ~ ±6.8 pt)
    "RL_EVAL_SEED": 2026,                      # Fixed validation wall seed (identical walls across all versions for paired progress tracking)
    "RL_EVAL_CHUNK_WALLS": max(1, min(_EVAL_CHUNK // 2, 8)),  # per device; conservative while RL XLA graphs stay resident
    "RL_EVAL_VS_BEST": False,                  # Optional extra 1v3 eval vs current rl_best.pkl (catches BC-overfitting)
    "RL_DEGRADE_PATIENCE": 3,                  # Warn (or early-stop if RL_EARLY_STOP=True) after K consecutive evals significantly below best
    "RL_EARLY_STOP": False,                    # If True, cleanly stop RL when degraded for RL_DEGRADE_PATIENCE evals (preserves rl_best.pkl)
    "RL_USE_BEST_FOR_DEPLOY": True,            # Prefer rl_best.pkl (highest periodic eval pt vs BC) over final weights if available

    # Phase 3 SD-TAP behavioral self-distillation
    "SDTAP_EPOCHS": 2,
    "SDTAP_BATCH_SIZE": 256,

    # Strict Duplicate 1v3 evaluation (pmap across ALL GPUs: EVAL_CHUNK_WALLS * N_DEVICES walls/chunk!)
    "EVAL_WALLS": 256,                         # 256 walls = 1,024 half-games on independent test seed (CI ~ ±4.8 pt)
    "EVAL_CHUNK_WALLS": _EVAL_CHUNK,
    "EVAL_TEST_SEED": 9999,                    # Disjoint from RL_EVAL_SEED=2026 to avoid validation-wall overfitting

    # Optional: Hugging Face Hub auto-sync (bind HF_TOKEN in Kaggle Add-ons -> Secrets!)
    "HF_DATASET_REPO": "myouo/tenhou-2026-v6-shards",
    "HF_DATASET_SUBDIR": "npz",                 # Store .npz shards under <repo>/npz/
    "HF_DATASET_PRIVATE": True,                 # True=private dataset repo, False=public
    "HF_AUTO_DOWNLOAD_SHARDS": False,           # Online RL does not need 108 shards; set True if retraining Phase 0
    "HF_AUTO_UPLOAD_SHARDS": True,              # Push newly converted shards + manifest.json to HF_DATASET_REPO
    "HF_UPLOAD_REPO": "myouo/sovereign-mahjong-v6",
    "HF_AUTO_DOWNLOAD_CKPTS": True,            # Auto-pull latest checkpoints from HF_UPLOAD_REPO at session start
    "HF_MODEL_PRIVATE": True,
    "SEED": 0,
}

PARQUET_DIR = TMP_ROOT / "tenhou_2026_mjai"
BASELINE_DIR = TMP_ROOT / "baselines" / "tanyaodojo"
SHARD_DIR = Path(CFG["PREEXISTING_SHARD_DIR"]) if CFG["PREEXISTING_SHARD_DIR"] else (WORK_ROOT / "v6_shards_2026")
CKPT_DIR = WORK_ROOT / "checkpoints" / "sovereign_v6"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-pull existing checkpoints from HF Model Hub so a fresh Kaggle session resumes seamlessly
if CFG.get("HF_AUTO_DOWNLOAD_CKPTS", True) and CFG.get("HF_UPLOAD_REPO"):
    try:
        from mahjong_rl.data.hf_sync import download_checkpoints_from_hf
        download_checkpoints_from_hf(CFG["HF_UPLOAD_REPO"], CKPT_DIR)
    except Exception as _exc:
        print(f"[HF Auto-Download Checkpoints] Skipped ({type(_exc).__name__}: {_exc})")

print(f"Detected Hardware: {N_DEVICES}x GPU ({VRAM_GB:.1f} GB/GPU = {N_DEVICES*VRAM_GB:.1f} GB total VRAM)")
print(f"Auto-Tuned Plan:   conv_batch={CFG['ROUND_BATCH_SIZE']} | belief_bs={CFG['BELIEF_BATCH_SIZE']} | bc_bs={CFG['BC_BATCH_SIZE']} | rl_envs={CFG['RL_NUM_ENVS']} | rl_mb={CFG['RL_MINIBATCH_SIZE']}")
print("SHARD_DIR:", SHARD_DIR)
print("CKPT_DIR: ", CKPT_DIR)

## 2.5 GPU 吞吐体检：`num_envs` batch sweep（先跑这个）

> 平台：Kaggle 2×T4。`num_envs` 是**全局** batch，每卡只有 `num_envs / N_DEVICES` 个环境。
> 本机单张 3060 在 B=32 已 compute-bound；T4 每卡更慢，但 16 envs/卡 仍可能偏小。

本 cell 依次跑 `num_envs ∈ CFG["PERF_BENCH_BATCHES"]`，每个配置 4 个 update。

> **Kaggle 2×T4 实测（2026-11）**：在 2×T4 上，**update 1 和 update 2 各包含一次编译**
> （分别约 210–250s 和 185s），只有 update 3 起才是稳态。用 `(upd2+upd3)/2`
> 会把吞吐低估约 2×。

| `num_envs` | 每卡 | update 3 稳态 |
|---|---|---|
| 32 | 16 | 1,174 sps |
| 64 | 32 | 1,486 sps |
| **128** | **64** | **1,766 sps** |

- 默认已经改成 `[128, 256]`，用来确认 128 之后是否还有收益。
- 若 128→256 明显上升：继续试 256；持平就停在 128。
- 编译缓存写入 `JAX_CACHE_DIR`；重复跑同一 shape 会快很多。
- 每个配置都要编译一次，3 个配置在 Kaggle 上约 **30–60 分钟**（冷缓存）；之后重跑会快很多。
- 需要 Phase 0 的 `bc_phase0c.pkl / belief_phase0a.pkl / grp_phase0b.pkl`（cell 4 会自动从 HF 拉取）。


In [ ]:
RUN_BATCH_SWEEP = True   # 单独运行本 cell 时改成 True
SWEEP_BATCHES = [int(b) for b in CFG.get("PERF_BENCH_BATCHES", [128, 256])]
SWEEP_UPDATES = int(CFG.get("PERF_BENCH_UPDATES", 4))
BENCH_ROOT = TMP_ROOT / "mahjax_batch_sweep"

if RUN_BATCH_SWEEP:
    import re
    import shutil
    import time

    _ckpt_src = Path(CFG["PREEXISTING_CKPT_DIR"]) if CFG["PREEXISTING_CKPT_DIR"] else CKPT_DIR
    _required = ["bc_phase0c.pkl", "belief_phase0a.pkl", "grp_phase0b.pkl"]
    _missing = [n for n in _required if not (_ckpt_src / n).exists()]
    if _missing:
        print(f"[SKIP] missing Phase-0 checkpoints in {_ckpt_src}: {_missing}")
        print("       Run Phase 0 first, or mount a Kaggle Dataset with the .pkl files.")
    else:
        BENCH_ROOT.mkdir(parents=True, exist_ok=True)
        _env = os.environ.copy()
        _env["JAX_COMPILATION_CACHE_DIR"] = str(JAX_CACHE_DIR)
        _env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
        _env["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
        results = []
        for B in SWEEP_BATCHES:
            if B % max(N_DEVICES, 1) != 0:
                print(f"[SKIP] num_envs={B} is not divisible by {N_DEVICES} GPUs")
                continue
            out = BENCH_ROOT / f"b{B}"
            if out.exists():
                shutil.rmtree(out)
            cmd = [
                sys.executable, "-m", "mahjong_rl.trainers.run_local_rl",
                "--ckpt-dir", str(_ckpt_src), "--out-dir", str(out),
                "--num-envs", str(B), "--num-steps", str(CFG["RL_NUM_STEPS"]),
                "--minibatch-size", str(CFG["RL_MINIBATCH_SIZE"]),
                "--num-updates", str(SWEEP_UPDATES), "--warmup-updates", "0",
                "--eval-every", "0", "--checkpoint-every", "0", "--log-every", "1",
                "--no-resume",
            ]
            print(f"\n===== num_envs={B} ({B // max(N_DEVICES, 1)}/GPU) =====", flush=True)
            t0 = time.perf_counter()
            sps_by_upd = {}
            proc = subprocess.Popen(
                cmd, cwd=str(REPO_DIR), env=_env,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1,
            )
            for line in proc.stdout:
                line = line.rstrip()
                if any(tok in line for tok in ("[RL ", "envs=", "Traceback", "Error", "error", "out of memory", "RESOURCE")):
                    print(line, flush=True)
                m = re.search(r"upd\s+(\d+)/\d+\s+\([0-9.]+s,\s*([\d,]+)\s+sps\)", line)
                if m:
                    sps_by_upd[int(m.group(1))] = float(m.group(2).replace(",", ""))
            rc = proc.wait()
            wall = time.perf_counter() - t0
            # Updates 1 and 2 each include a one-time compile on 2xT4; only >=3 is steady.
            steady = [v for k, v in sorted(sps_by_upd.items()) if k >= 3]
            steady_sps = sum(steady) / len(steady) if steady else float("nan")
            results.append({
                "num_envs": B, "per_gpu": B // max(N_DEVICES, 1),
                "upd1_sps": sps_by_upd.get(1, float("nan")),
                "upd2_sps": sps_by_upd.get(2, float("nan")),
                "steady_sps": steady_sps, "wall_min": wall / 60.0, "rc": rc,
            })
            sps_str = " ".join(f"upd{k}={v:,.0f}" for k, v in sorted(sps_by_upd.items()))
            print(f"  -> rc={rc} wall={wall / 60:.1f} min | {sps_str} | steady={steady_sps:,.0f} sps", flush=True)

        if results:
            import pandas as pd
            df = pd.DataFrame(results)
            print("\n===== batch sweep summary =====")
            print(df.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))
            ok = df[(df["rc"] == 0)].dropna(subset=["steady_sps"])
            if len(ok) == 0:
                print("\n[!] 所有 sweep 都失败了：先确认 --ckpt-dir 指向有效的 Phase-0 权重，"
                      "并检查上面的 Traceback。")
            else:
                best = ok.loc[ok["steady_sps"].idxmax()]
                print(
                    f"\n[推荐] num_envs={int(best['num_envs'])} ({int(best['per_gpu'])}/GPU) "
                    f"-> {best['steady_sps']:,.0f} sps"
                )
                if N_DEVICES > 1:
                    print(f"        把 cell 4 的 _PER_DEV_RL_ENVS 改成 {int(best['per_gpu'])} 即可。")
                print(
                    "        注意：1 个 update 的数据量 = num_envs * num_steps。若要维持总环境步数不变，\n"
                    "        请把 RL_NUM_UPDATES（以及 WARMUP / PFSP / EVAL 这些以 update 计的间隔）\n"
                    "        按 num_envs 的变化比例一起调整。"
                )
            try:
                import matplotlib.pyplot as plt
                ax = ok.plot.bar(x="num_envs", y="steady_sps", legend=False, figsize=(6, 3.5))
                ax.set_xlabel("num_envs (global)")
                ax.set_ylabel("steady sps")
                ax.set_title(f"Batch sweep ({N_DEVICES}x GPU)")
                ax.grid(True, axis="y", alpha=0.3)
                plt.tight_layout()
                plt.show()
            except Exception as _exc:
                print("plot skipped:", _exc)
else:
    print("Skipped batch sweep (set RUN_BATCH_SWEEP = True).")


## 3. 下载 2026 天凤凤桌 MJAI Parquet 数据集与 TanyaoDojo 参考基线

In [ ]:
import subprocess
import pyarrow.parquet as pq

def _download(url: str, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size > 1024:
        print(f"  [skip] {dst.name} already exists ({dst.stat().st_size / 1e6:.1f} MB)")
        return
    print(f"  [fetch] {dst.name} ...")
    subprocess.run(["curl", "-fL", "--retry", "3", "-C", "-", url, "-o", str(dst)], check=True)

if CFG["RUN_DOWNLOAD"]:
    print("=== Downloading 2026 Tenhou Phoenix MJAI Parquet (hhim8826/tenhou-houou-mjai) ===")
    for fname in ("tenhou-new-00000.parquet", "tenhou-new-00001.parquet"):
        _download(
            f"https://huggingface.co/datasets/hhim8826/tenhou-houou-mjai/resolve/main/data/{fname}",
            PARQUET_DIR / fname,
        )

    print("\n=== Downloading Twu31/TanyaoDojo Reference Baseline (rl_oracle_800m.pkl) ===")
    for fname in ("README.md", "rl_oracle_800m.pkl"):
        _download(
            f"https://huggingface.co/Twu31/TanyaoDojo/resolve/main/{fname}",
            BASELINE_DIR / fname,
        )

    total_rows = 0
    for p in sorted(PARQUET_DIR.glob("*.parquet")):
        n = pq.ParquetFile(p).metadata.num_rows
        total_rows += n
        print(f"  Verified {p.name}: {n:,} games ({p.stat().st_size / 1e6:.1f} MB)")
    print(f"Total available 4p Phoenix hanchan games: {total_rows:,}")

## 4. GPU 批量无损转换：MJAI Parquet → Sovereign-Mahjong V6 `.npz` 分片

In [ ]:
import json
from mahjong_rl.data.hf_sync import download_v6_shards_from_hf, upload_v6_shards_to_hf
from mahjong_rl.data.mjai_parquet_converter import convert_mjai_parquets_to_v6_shards

def _check_manifest_complete(shard_dir: Path, start_game: int, target_games):
    shards = sorted(shard_dir.glob("shard_*.npz"))
    m_path = shard_dir / "manifest.json"
    m = json.loads(m_path.read_text(encoding="utf-8")) if m_path.exists() else None
    ok = (
        m is not None
        and bool(m.get("completed", True))
        and int(m.get("start_game", 0)) == int(start_game)
        and (target_games is None or int(m.get("games_processed", 0)) >= int(target_games))
        and len(shards) >= max(len(m.get("shards_written", [])), 1)
    )
    return shards, m_path, m, ok

target_games = CFG["MAX_GAMES"]
existing_shards, manifest_path, manifest, manifest_complete = _check_manifest_complete(
    SHARD_DIR, CFG["START_GAME"], target_games
)

# Optional: pull existing V6 shards from Hugging Face Dataset first (skip or resume seamlessly!)
if (
    CFG.get("HF_DATASET_REPO")
    and CFG.get("HF_AUTO_DOWNLOAD_SHARDS", True)
    and not manifest_complete
    and not CFG.get("FORCE_RECONVERT", False)
):
    try:
        download_v6_shards_from_hf(
            CFG["HF_DATASET_REPO"],
            SHARD_DIR,
            path_in_repo=CFG.get("HF_DATASET_SUBDIR", "npz"),
        )
        existing_shards, manifest_path, manifest, manifest_complete = _check_manifest_complete(
            SHARD_DIR, CFG["START_GAME"], target_games
        )
    except Exception as _exc:
        print(f"[HF Download Shards] Skipped ({type(_exc).__name__}: {_exc})")

need_convert = (
    CFG["RUN_CONVERT"]
    and (CFG.get("FORCE_RECONVERT", False) or not manifest_complete)
    and not (CFG.get("ALLOW_PARTIAL_SHARDS", False) and bool(existing_shards))
)

if need_convert:
    if CFG.get("FORCE_RECONVERT", False) and (existing_shards or manifest_path.exists()):
        print(f"[Convert] FORCE_RECONVERT=True — removing {len(existing_shards)} existing shard(s) ...")
        for _p in list(SHARD_DIR.glob("shard_*.npz")) + list(SHARD_DIR.glob(".shard_*.tmp.npz")):
            _p.unlink(missing_ok=True)
        manifest_path.unlink(missing_ok=True)
        (SHARD_DIR / "progress.json").unlink(missing_ok=True)
    elif existing_shards:
        print(
            f"[Convert] Keeping {len(existing_shards)} existing shard(s) and RESUMING automatically "
            f"from where conversion stopped until MAX_GAMES={target_games} ..."
        )
    parquets = sorted(PARQUET_DIR.glob("*.parquet"))
    assert parquets, f"No parquet files found in {PARQUET_DIR}"
    conv_res = convert_mjai_parquets_to_v6_shards(
        parquets,
        out_dir=SHARD_DIR,
        max_games=CFG["MAX_GAMES"],
        start_game=CFG["START_GAME"],
        hanchan_only=True,
        round_batch_size=CFG["ROUND_BATCH_SIZE"],
        max_steps_per_round=CFG["MAX_STEPS_PER_ROUND"],
        ron_k_max=CFG["RON_K_MAX"],
        shard_size=CFG["SHARD_SIZE"],
        storage_dtype="float16",
        include_oracle=True,
        compress=True,
        compresslevel=CFG["COMPRESS_LEVEL"],
        multi_gpu=True,
        resume=not CFG.get("FORCE_RECONVERT", False),
        verbose=True,
    )
    print("\nConversion Summary:")
    print(json.dumps({k: v for k, v in conv_res.items() if k != "shards_written"}, indent=2))
    assert conv_res["clean_round_rate"] >= 0.995, f"Unexpected desync rate: {conv_res['clean_round_rate']}"
    assert conv_res["ron_overflow_batches"] == 0, (
        f"ron_k_max overflowed {conv_res['ron_overflow_batches']} times; increase CFG['RON_K_MAX']!"
    )
    if CFG.get("HF_DATASET_REPO") and CFG.get("HF_AUTO_UPLOAD_SHARDS", True):
        upload_v6_shards_to_hf(
            SHARD_DIR,
            CFG["HF_DATASET_REPO"],
            path_in_repo=CFG.get("HF_DATASET_SUBDIR", "npz"),
            private=CFG.get("HF_DATASET_PRIVATE", True),
        )
elif manifest_complete:
    print(
        f"Using {len(existing_shards)} verified V6 shards from {SHARD_DIR} "
        f"(games={manifest.get('games_processed')}, steps={manifest.get('steps_written'):,}, "
        f"clean={manifest.get('clean_round_rate', 0.0)*100:.2f}%)"
    )
else:
    print(
        f"WARNING: Using {len(existing_shards)} partial/unverified shards from {SHARD_DIR} "
        f"(RUN_CONVERT={CFG['RUN_CONVERT']}, ALLOW_PARTIAL_SHARDS={CFG.get('ALLOW_PARTIAL_SHARDS', False)})"
    )

all_shards = sorted(SHARD_DIR.glob("shard_*.npz"))
_need_shards = (
    CFG.get("FORCE_RETRAIN_GRP", False)
    or CFG.get("FORCE_RETRAIN_BELIEF", False)
    or CFG.get("FORCE_RETRAIN_BC", False)
    or not (CKPT_DIR / "bc_phase0c.pkl").exists()
)
if len(all_shards) <= CFG["VAL_SHARDS"]:
    if _need_shards:
        raise AssertionError(
            f"Found only {len(all_shards)} shards in {SHARD_DIR}; enable HF_AUTO_DOWNLOAD_SHARDS=True or RUN_CONVERT=True."
        )
    # Online RL mode: optionally grab just 6 shards for Phase 3 SD-TAP if HF_DATASET_REPO is configured
    if CFG.get("RUN_PHASE3_SDTAP", True) and CFG.get("HF_DATASET_REPO"):
        try:
            download_v6_shards_from_hf(
                CFG["HF_DATASET_REPO"],
                SHARD_DIR,
                path_in_repo=CFG.get("HF_DATASET_SUBDIR", "npz"),
                max_shards=CFG["VAL_SHARDS"] + 2,
            )
            all_shards = sorted(SHARD_DIR.glob("shard_*.npz"))
        except Exception as _exc:
            print(f"[HF Light Shard Pull for SD-TAP] Skipped ({type(_exc).__name__}: {_exc})")

if len(all_shards) > CFG["VAL_SHARDS"]:
    train_shards = all_shards[: -CFG["VAL_SHARDS"]]
    val_shards = all_shards[-CFG["VAL_SHARDS"] :]
    print(f"Strict split: {len(train_shards)} train shards | {len(val_shards)} held-out val shards")
else:
    train_shards, val_shards = [], []
    print("[Online RL Mode] No local .npz shards required — Phase 0 checkpoints will be loaded directly.")

## 5. Phase 0-B：训练 Birkhoff `(4, 4)` `GRP_base` 并检验零和守恒门控

In [ ]:
from dataclasses import asdict
import pickle
import time
import jax

from mahjong_rl.config import GRPConfig
from mahjong_rl.models.grp import GRPNet
from mahjong_rl.trainers.run_phase0 import evaluate_grp_val
from mahjong_rl.trainers.train_grp import train_grp

grp_ckpt_path = (
    Path(CFG["PREEXISTING_CKPT_DIR"]) / "grp_phase0b.pkl"
    if CFG["PREEXISTING_CKPT_DIR"]
    else CKPT_DIR / "grp_phase0b.pkl"
)

if (CFG["RUN_PHASE0"] and CFG.get("FORCE_RETRAIN_GRP", False)) or not grp_ckpt_path.exists():
    t0 = time.perf_counter()
    grp_cfg = GRPConfig(num_epochs=CFG["GRP_EPOCHS"], batch_size=CFG["GRP_BATCH_SIZE"])
    grp_model, grp_params, grp_hist = train_grp(train_shards, cfg=grp_cfg, seed=CFG["SEED"])
    grp_val = evaluate_grp_val(grp_model, grp_params, val_shards)
    with open(CKPT_DIR / "grp_phase0b.pkl", "wb") as f:
        pickle.dump(jax.device_get({"config": asdict(grp_cfg), "params": grp_params, "val": grp_val}), f)
    print(f"Phase 0-B finished in {time.perf_counter() - t0:.1f}s")
else:
    print(f"[Phase 0-B GRP] Skipping training — loading existing checkpoint: {grp_ckpt_path} (set FORCE_RETRAIN_GRP=True to retrain)")
    with open(grp_ckpt_path, "rb") as f:
        d = pickle.load(f)
    grp_cfg = GRPConfig(**d["config"])
    grp_model = GRPNet(config=grp_cfg)
    grp_params = jax.device_put(d["params"])
    grp_val = d.get("val") or evaluate_grp_val(grp_model, grp_params, val_shards)

print("Phase 0-B Held-Out Validation Metrics:")
print(json.dumps(grp_val, indent=2))
assert grp_val["val_grp_birkhoff_mean_err"] < 1e-3, "Birkhoff mean doubly-stochastic gate failed!"
assert grp_val["val_grp_birkhoff_max_err"] < 1e-2, "Birkhoff worst-case doubly-stochastic gate failed!"
assert grp_val["val_grp_7d_zero_sum_mean_err"] < 1e-3, "7d mean zero-sum gate failed!"


## 6. Phase 0-A：训练 36× 等变 `BeliefNet` + 独立留出集温度校准 `T_cal`

In [ ]:
from mahjong_rl.config import BeliefConfig
from mahjong_rl.models.belief import BeliefNet
from mahjong_rl.trainers.run_phase0 import evaluate_belief_val
from mahjong_rl.trainers.train_belief import train_belief

bel_ckpt_path = (
    Path(CFG["PREEXISTING_CKPT_DIR"]) / "belief_phase0a.pkl"
    if CFG["PREEXISTING_CKPT_DIR"]
    else CKPT_DIR / "belief_phase0a.pkl"
)

if (CFG["RUN_PHASE0"] and CFG.get("FORCE_RETRAIN_BELIEF", False)) or not bel_ckpt_path.exists():
    t0 = time.perf_counter()
    bel_cfg = BeliefConfig(num_epochs=CFG["BELIEF_EPOCHS"], batch_size=CFG["BELIEF_BATCH_SIZE"], seed=CFG["SEED"])
    bel_model, bel_params, belief_t_cal, bel_hist = train_belief(
        train_shards, cfg=bel_cfg, val_shard_paths=val_shards, suit_augment=True
    )
    bel_val = evaluate_belief_val(bel_model, bel_params, belief_t_cal, val_shards)
    with open(CKPT_DIR / "belief_phase0a.pkl", "wb") as f:
        pickle.dump(
            jax.device_get({"config": asdict(bel_cfg), "params": bel_params, "t_cal": float(belief_t_cal), "val": bel_val}),
            f,
        )
    print(f"Phase 0-A finished in {time.perf_counter() - t0:.1f}s")
else:
    print(f"[Phase 0-A Belief] Skipping training — loading existing checkpoint: {bel_ckpt_path} (set FORCE_RETRAIN_BELIEF=True to retrain)")
    with open(bel_ckpt_path, "rb") as f:
        d = pickle.load(f)
    bel_cfg = BeliefConfig(**d["config"])
    bel_model = BeliefNet(config=bel_cfg)
    bel_params = jax.device_put(d["params"])
    belief_t_cal = float(d["t_cal"])
    bel_val = d.get("val") or evaluate_belief_val(bel_model, bel_params, belief_t_cal, val_shards)

print("Phase 0-A Held-Out Validation Metrics:")
print(json.dumps(bel_val, indent=2))


## 7. Phase 0-C：严格同构 `SovereignPolicyNet` 行为克隆（含 `beta_risk / beta_offense`）

In [ ]:
from mahjong_rl.config import BCConfig, ModelConfig
from mahjong_rl.models.equivariant import SovereignPolicyNet
from mahjong_rl.trainers.run_phase0 import evaluate_bc_val
from mahjong_rl.trainers.train_bc import train_bc

bc_ckpt_path = (
    Path(CFG["PREEXISTING_CKPT_DIR"]) / "bc_phase0c.pkl"
    if CFG["PREEXISTING_CKPT_DIR"]
    else CKPT_DIR / "bc_phase0c.pkl"
)

if (CFG["RUN_PHASE0"] and CFG.get("FORCE_RETRAIN_BC", False)) or not bc_ckpt_path.exists():
    t0 = time.perf_counter()
    model_cfg = ModelConfig()
    bc_cfg = BCConfig(
        model=model_cfg,
        num_epochs=CFG["BC_EPOCHS"],
        batch_size=CFG["BC_BATCH_SIZE"],
        suit_augment=True,
        seed=CFG["SEED"],
    )
    pol_model, bc_params, bc_hist = train_bc(
        train_shards,
        belief_model=bel_model,
        belief_params=bel_params,
        grp_model=grp_model,
        grp_params=grp_params,
        cfg=bc_cfg,
        belief_t_cal=belief_t_cal,
    )
    bc_val = evaluate_bc_val(
        pol_model, bc_params, bel_model, bel_params, belief_t_cal, grp_model, grp_params, val_shards
    )
    with open(CKPT_DIR / "bc_phase0c.pkl", "wb") as f:
        pickle.dump(
            jax.device_get({"model_config": asdict(model_cfg), "params": bc_params, "t_cal": float(belief_t_cal), "val": bc_val}),
            f,
        )
    print(f"Phase 0-C finished in {time.perf_counter() - t0:.1f}s")
else:
    print(f"[Phase 0-C BC] Skipping training — loading existing checkpoint: {bc_ckpt_path} (set FORCE_RETRAIN_BC=True to retrain)")
    with open(bc_ckpt_path, "rb") as f:
        d = pickle.load(f)
    model_cfg = ModelConfig(**d["model_config"])
    pol_model = SovereignPolicyNet(config=model_cfg)
    bc_params = jax.device_put(d["params"])
    bc_val = d.get("val") or evaluate_bc_val(
        pol_model, bc_params, bel_model, bel_params, belief_t_cal, grp_model, grp_params, val_shards
    )

print("Phase 0-C Held-Out Validation Metrics:")
print(json.dumps(bc_val, indent=2))


## 7.5 零重编译固定牌山自测器 & 分版本存档管理器

把候选/基线网络参数剥离出闭包作为 `jax.pmap / jax.jit` 动态输入（整个训练周期仅编译一次 XLA 计算图）：
- 每 `RL_VERSION_EVERY` 步在 `CKPT_DIR` 下保存独立轻量快照 `rl_upd_XXXX.pkl`（约 6.7 MB，可直接推理/部署）；
- 每 `RL_EVAL_EVERY` 步在固定验证牌山（`RL_EVAL_SEED=2026`）上跑 4 席配对 Duplicate 1v3（1× 当前 RL vs 3× 冻结 BC），写入 `rl_eval_history.json` 并自动维护最优版本 `rl_best.pkl`；
- 兼容工具 `11.7` 离线批量回测历史版本与 `outer_snapshots`。

In [ ]:
import gc
import json
import os
import pickle
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import jax
import jax.numpy as jnp
from jax import lax
import numpy as np

import mahjax
from mahjax.common.action import Action
from mahjong_rl.algo.legal_ron import compute_analytical_trinity, extract_table_snapshot
from mahjong_rl.algo.sovereign_v6 import (
    NEG_INF,
    masked_categorical_log_probs,
    replicate_for_pmap,
    shard_for_pmap,
)
from mahjong_rl.algo.utility import (
    W_PHOENIX_7D,
    build_opponent_utility_features,
    make_homogeneous_table,
    rotate_utility_matrix_ego,
)
from mahjong_rl.config import EvalConfig
from mahjong_rl.features.obs_2d import clean_auto_reset_state
from mahjong_rl.features.obs_v6 import encode_obs_v6


class _EarlyStopRL(Exception):
    """Signal raised after checkpointing when degradation patience is exhausted."""


def _atomic_dump_pkl(path: Path, obj: Any) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp")
    with open(tmp, "wb") as f:
        pickle.dump(jax.device_get(obj), f)
    os.replace(tmp, path)
    return path


def _atomic_write_json(path: Path, obj: Any) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)
    return path


class CompiledDuplicate1v3Evaluator:
    """Zero-recompilation fixed-wall Duplicate 1v3 evaluator (1 candidate vs 3 baseline).

    Policy and Belief weights are explicit dynamic arguments to `pmap`/`jit` instead of
    closed-over constants, so arbitrarily many checkpoints can be benchmarked after a
    single XLA compilation. Bit-exact with `mahjong_rl.eval.duplicate_eval.evaluate_duplicate_1v3`.
    """

    def __init__(
        self,
        *,
        policy_model: Any,
        belief_model: Any,
        grp_model: Any,
        grp_base_params: Any,
        eval_cfg: EvalConfig = EvalConfig(),
        w_rule: Any = W_PHOENIX_7D,
        opp_rules_known: bool = False,
        belief_t_cal: float = 1.0,
        greedy: bool = True,
        chunk_walls: int = 8,
    ):
        self.eval_cfg = eval_cfg
        self.chunk_walls = max(1, int(chunk_walls))
        self.n_devices = jax.local_device_count()

        env = mahjax.make(
            eval_cfg.env_name,
            round_mode=eval_cfg.round_mode,
            observe_type="dict",
            round_end="confirm",
        )
        num_players = env.num_players
        self.num_players = num_players

        # Keep frozen shared data as host NumPy arrays so pmap replicates cleanly across GPUs
        grp_np = jax.tree_util.tree_map(lambda x: np.asarray(jax.device_get(x)), grp_base_params)
        w_tab_np = np.asarray(jax.device_get(make_homogeneous_table(w_rule)), dtype=np.float32)
        w_rule_np = np.asarray(jax.device_get(w_rule), dtype=np.float32)
        is_known_np = np.full((3,), 1.0 if opp_rules_known else 0.0, dtype=np.float32)
        rank_pts_np = np.asarray(eval_cfg.rank_points, dtype=np.float32)
        t_cal_f = float(belief_t_cal)
        grp_fn = lambda gf: grp_model.apply(grp_np, gf)

        def _act_from_encoded(obs, tbl, w_me, w_opp_5d, legal_mask, key, pol_p, bel_p):
            b_out = belief_model.apply(
                bel_p,
                planes=obs["planes"],
                scalars=obs["scalars"],
                river_tokens=obs["river_tokens"],
                river_tile_ids=obs["river_tile_ids"],
                river_valid_mask=obs["river_valid_mask"],
                w_opp_5d=w_opp_5d,
                temperature_cal=t_cal_f,
            )
            du_ron_3opp, du_ron_87, du_agari_3, du_ryu, call_threat = compute_analytical_trinity(
                s_table=tbl,
                p_deal_3opp=b_out["p_deal"],
                p_score_bins=b_out["p_score_bin"],
                p_call_threat_3=b_out["p_call_threat"],
                p_tenpai_3=b_out["p_tenpai"],
                w_me=w_me,
                last_draw_type=obs["last_draw_type"],
                is_last_draw_red=obs["is_last_draw_red"],
                grp_fn=grp_fn,
            )
            logits = policy_model.apply(
                pol_p,
                planes=obs["planes"],
                delta_u_ron_3opp=du_ron_3opp,
                call_threat_plane=call_threat,
                scalars=obs["scalars"],
                river_tokens=obs["river_tokens"],
                river_tile_ids=obs["river_tile_ids"],
                river_valid_mask=obs["river_valid_mask"],
                last_draw_type=obs["last_draw_type"],
                is_last_draw_red=obs["is_last_draw_red"],
                target_type=obs["target_type"],
                target_is_red=obs["target_is_red"],
                w_ego=w_me,
                delta_u_agari_3=du_agari_3,
                delta_u_ryuukyoku=du_ryu,
                delta_u_ron_87=du_ron_87,
                legal_action_mask=legal_mask,
            )
            if greedy:
                return jnp.argmax(jnp.where(legal_mask, logits, NEG_INF)).astype(jnp.int32)
            log_p = masked_categorical_log_probs(logits, legal_mask)
            return jax.random.categorical(key, log_p).astype(jnp.int32)

        def play_single_seat(c_pol, c_bel, b_pol, b_bel, init_state, agent_seat, wall_step_seed):
            def step_body(carry, step_idx):
                state, rng = carry
                done = state.terminated | state.truncated
                rng, k1, k2 = jax.random.split(rng, 3)
                kyoku_id = state.round_state.round.astype(jnp.int32) * 64 + jnp.minimum(
                    state.round_state.honba.astype(jnp.int32), 63
                )
                k_step = jax.random.fold_in(wall_step_seed, kyoku_id)

                is_dummy = state.legal_action_mask[Action.DUMMY]
                cp_state = state.current_player.astype(jnp.int32)

                clean_st = clean_auto_reset_state(state)
                cp_obs = clean_st.current_player.astype(jnp.int32)
                legal_mask = clean_st.legal_action_mask.astype(jnp.bool_)
                obs = encode_obs_v6(clean_st)
                tbl = extract_table_snapshot(clean_st)
                w_ego_4x4 = rotate_utility_matrix_ego(w_tab_np, cp_obs)
                w_me = w_ego_4x4[0]
                w_opp_5d = build_opponent_utility_features(
                    w_ego_4x4[1:],
                    is_known_np,
                    rng_key=None,
                    dropout_prob=0.0,
                    w_table_default=w_rule_np,
                )

                a_cand = _act_from_encoded(obs, tbl, w_me, w_opp_5d, legal_mask, k1, c_pol, c_bel)
                a_base = _act_from_encoded(obs, tbl, w_me, w_opp_5d, legal_mask, k2, b_pol, b_bel)
                policy_act = jnp.where(cp_state == agent_seat, a_cand, a_base)
                action = jnp.where(is_dummy, jnp.int32(Action.DUMMY), policy_act)

                next_state = lax.cond(done, lambda: state, lambda: env.step(state, action, k_step))

                hand_ended = (
                    ~done
                    & next_state.round_state.terminated_round
                    & ~state.round_state.terminated_round
                )
                p = next_state.players
                won = p.has_won
                riichi = p.riichi
                melded = ~p.is_hand_concealed
                rewards = next_state.rewards.astype(jnp.float32)
                is_hora = won.any()
                is_ron = is_hora & ((rewards < 0.0).sum() == 1)
                dealt_in_mask = is_ron & (rewards < 0.0)

                cand_won = hand_ended & won[agent_seat]
                cand_dealt_in = hand_ended & dealt_in_mask[agent_seat]
                cand_riichi = hand_ended & riichi[agent_seat]
                cand_melded = hand_ended & melded[agent_seat]

                ev = {
                    "hands": hand_ended.astype(jnp.int32),
                    "won": cand_won.astype(jnp.int32),
                    "dealt_in": cand_dealt_in.astype(jnp.int32),
                    "riichi": cand_riichi.astype(jnp.int32),
                    "melded": cand_melded.astype(jnp.int32),
                    "won_pts": jnp.where(cand_won, rewards[agent_seat], 0.0),
                    "dealt_in_pts": jnp.where(cand_dealt_in, -rewards[agent_seat], 0.0),
                }
                return (next_state, rng), ev

            (final_state, _), events = lax.scan(
                step_body,
                (init_state, wall_step_seed),
                jnp.arange(eval_cfg.max_steps, dtype=jnp.int32),
            )
            totals = jax.tree.map(lambda x: x.sum(axis=0), events)

            scores = final_state.round_state.score.astype(jnp.float32)
            init_wind = final_state.round_state.init_wind.astype(jnp.int32)
            my_score = scores[agent_seat]
            my_wind = init_wind[agent_seat]
            ahead = (scores > my_score) | ((scores == my_score) & (init_wind < my_wind))
            rank_idx = ahead.sum().astype(jnp.int32)
            pt = jnp.asarray(rank_pts_np, dtype=jnp.float32)[rank_idx]
            finished = (final_state.terminated | final_state.truncated).astype(jnp.int32)
            return totals, rank_idx, pt, finished

        def play_duplicate_wall(c_pol, c_bel, b_pol, b_bel, init_key, wall_step_seed):
            init_state = env.init(init_key)
            seats = jnp.arange(num_players, dtype=jnp.int32)
            return jax.vmap(
                lambda s: play_single_seat(c_pol, c_bel, b_pol, b_bel, init_state, s, wall_step_seed)
            )(seats)

        self._run_single_dev = jax.jit(
            lambda c_pol, c_bel, b_pol, b_bel, ik, ss: jax.vmap(
                lambda k1, k2: play_duplicate_wall(c_pol, c_bel, b_pol, b_bel, k1, k2)
            )(ik, ss)
        )
        self._run_pmap = (
            jax.pmap(
                lambda c_pol, c_bel, b_pol, b_bel, ik, ss: jax.vmap(
                    lambda k1, k2: play_duplicate_wall(c_pol, c_bel, b_pol, b_bel, k1, k2)
                )(ik, ss)
            )
            if self.n_devices > 1
            else None
        )

        rng = jax.random.PRNGKey(eval_cfg.seed)
        k_init, k_step = jax.random.split(rng)
        self.init_keys = jax.random.split(k_init, eval_cfg.num_walls)
        self.step_seeds = jax.random.split(k_step, eval_cfg.num_walls)
        # Strong-reference cache (base_pol_ref, base_bel_ref, rep_b_pol, rep_b_bel)
        self._cached_base_pmap = None

    @staticmethod
    def _to_host_np(tree: Any) -> Any:
        return jax.tree_util.tree_map(lambda x: np.asarray(jax.device_get(x)), tree)

    def evaluate(
        self,
        cand_policy_params: Any,
        cand_belief_params: Any,
        base_policy_params: Any,
        base_belief_params: Any,
    ) -> Dict[str, Any]:
        t0 = time.perf_counter()
        num_walls = int(self.eval_cfg.num_walls)
        use_pmap = (self._run_pmap is not None) and (num_walls >= self.n_devices)
        if use_pmap:
            per_dev_w = max(1, min(self.chunk_walls, num_walls // self.n_devices))
            step_w = per_dev_w * self.n_devices
        else:
            step_w = max(1, min(self.chunk_walls, num_walls))

        rep_c_pol, rep_c_bel = None, None
        rep_b_pol, rep_b_bel = None, None
        single_c_pol, single_c_bel = None, None
        single_b_pol, single_b_bel = None, None

        totals_list, ranks_list, pts_list, fin_list = [], [], [], []
        try:
            for start in range(0, num_walls, step_w):
                end = min(start + step_w, num_walls)
                cnt = end - start
                if self._run_pmap is not None and cnt >= self.n_devices and (cnt % self.n_devices == 0):
                    if rep_c_pol is None:
                        rep_c_pol = replicate_for_pmap(cand_policy_params, self.n_devices)
                        rep_c_bel = replicate_for_pmap(cand_belief_params, self.n_devices)
                    if rep_b_pol is None:
                        if (
                            self._cached_base_pmap is not None
                            and self._cached_base_pmap[0] is base_policy_params
                            and self._cached_base_pmap[1] is base_belief_params
                        ):
                            _, _, rep_b_pol, rep_b_bel = self._cached_base_pmap
                        else:
                            rep_b_pol = replicate_for_pmap(base_policy_params, self.n_devices)
                            rep_b_bel = replicate_for_pmap(base_belief_params, self.n_devices)
                            self._cached_base_pmap = (
                                base_policy_params,
                                base_belief_params,
                                rep_b_pol,
                                rep_b_bel,
                            )
                    ik = shard_for_pmap(
                        self.init_keys[start:end].reshape(
                            (self.n_devices, cnt // self.n_devices) + self.init_keys.shape[1:]
                        ),
                        self.n_devices,
                    )
                    ss = shard_for_pmap(
                        self.step_seeds[start:end].reshape(
                            (self.n_devices, cnt // self.n_devices) + self.step_seeds.shape[1:]
                        ),
                        self.n_devices,
                    )
                    t_c, r_c, p_c, f_c = jax.device_get(
                        self._run_pmap(rep_c_pol, rep_c_bel, rep_b_pol, rep_b_bel, ik, ss)
                    )
                    t_c = {k: np.asarray(v).reshape((cnt,) + np.asarray(v).shape[2:]) for k, v in t_c.items()}
                    r_c = np.asarray(r_c, dtype=np.int32).reshape((cnt, self.num_players))
                    p_c = np.asarray(p_c, dtype=np.float64).reshape((cnt, self.num_players))
                    f_c = np.asarray(f_c, dtype=np.float64).reshape((cnt, self.num_players))
                else:
                    if single_c_pol is None:
                        single_c_pol = self._to_host_np(cand_policy_params)
                        single_c_bel = self._to_host_np(cand_belief_params)
                        single_b_pol = self._to_host_np(base_policy_params)
                        single_b_bel = self._to_host_np(base_belief_params)
                    t_c, r_c, p_c, f_c = jax.device_get(
                        self._run_single_dev(
                            single_c_pol,
                            single_c_bel,
                            single_b_pol,
                            single_b_bel,
                            self.init_keys[start:end],
                            self.step_seeds[start:end],
                        )
                    )
                    r_c = np.asarray(r_c, dtype=np.int32)
                    p_c = np.asarray(p_c, dtype=np.float64)
                    f_c = np.asarray(f_c, dtype=np.float64)
                totals_list.append(t_c)
                ranks_list.append(r_c)
                pts_list.append(p_c)
                fin_list.append(f_c)
        finally:
            del rep_c_pol, rep_c_bel, single_c_pol, single_c_bel, single_b_pol, single_b_bel

        totals = {k: np.concatenate([np.asarray(t[k]) for t in totals_list], axis=0) for k in totals_list[0]}
        rank_idxs_np = np.concatenate(ranks_list, axis=0)
        pts_np = np.concatenate(pts_list, axis=0)
        finished_np = np.concatenate(fin_list, axis=0)

        wall_done = finished_np.all(axis=1)
        finished_wall_ratio = float(wall_done.mean())
        eval_mask = wall_done if bool(wall_done.any()) else np.ones((num_walls,), dtype=bool)
        wall_mean_pts = pts_np[eval_mask].mean(axis=1)
        pt_mean = float(wall_mean_pts.mean())
        n_eval_w = int(wall_mean_pts.shape[0])
        pt_se = float(wall_mean_pts.std(ddof=1) / np.sqrt(max(n_eval_w, 1))) if n_eval_w > 1 else 0.0
        pt_ci95 = 1.96 * pt_se
        eval_ranks_np = rank_idxs_np[eval_mask]

        total_games = int(rank_idxs_np.size)
        total_hands = float(np.asarray(totals["hands"]).sum())
        denom_hands = max(total_hands, 1.0)
        total_won = float(np.asarray(totals["won"]).sum())
        total_dealt = float(np.asarray(totals["dealt_in"]).sum())

        return {
            "total_games": float(total_games),
            "total_hands": total_hands,
            "finished_game_ratio": float(finished_np.mean()),
            "finished_wall_ratio": finished_wall_ratio,
            "tenhou_pt_mean": pt_mean,
            "tenhou_pt_ci95": pt_ci95,
            "avg_rank": float(eval_ranks_np.mean() + 1.0),
            "rank_1_rate": float((eval_ranks_np == 0).mean()),
            "rank_2_rate": float((eval_ranks_np == 1).mean()),
            "rank_3_rate": float((eval_ranks_np == 2).mean()),
            "rank_4_rate": float((eval_ranks_np == 3).mean()),
            "hora_rate": total_won / denom_hands,
            "deal_in_rate": total_dealt / denom_hands,
            "riichi_rate": float(np.asarray(totals["riichi"]).sum()) / denom_hands,
            "meld_rate": float(np.asarray(totals["melded"]).sum()) / denom_hands,
            "avg_win_pts": float(np.asarray(totals["won_pts"]).sum()) / max(total_won, 1.0),
            "avg_deal_in_pts": float(np.asarray(totals["dealt_in_pts"]).sum()) / max(total_dealt, 1.0),
            "eval_wall_s": round(time.perf_counter() - t0, 2),
            "wall_mean_pts": [round(float(x), 3) for x in wall_mean_pts.tolist()],
        }


class VersionedEvalTracker:
    """Saves lightweight rl_upd_XXXX.pkl snapshots, tracks rl_best.pkl, and logs eval curves."""

    def __init__(
        self,
        *,
        ckpt_dir: Path,
        policy_model: Any,
        belief_model: Any,
        grp_model: Any,
        model_cfg: Any,
        bel_cfg: Any,
        grp_cfg: Any,
        grp_params: Any,
        bc_params: Any,
        bel_params: Any,
        belief_t_cal: float,
        target_updates: int,
        warmup_updates: int = 0,
        checkpoint_every: int = 50,
        version_every: int = 100,
        eval_every: int = 100,
        eval_walls: int = 32,
        eval_seed: int = 2026,
        eval_chunk_walls: int = 8,
        eval_vs_best: bool = False,
        degrade_patience: int = 3,
        early_stop: bool = False,
        hf_upload_repo: str = "",
        hf_model_private: bool = True,
        reset_history: bool = False,
    ):
        from dataclasses import asdict as _asdict

        self.ckpt_dir = Path(ckpt_dir)
        self.policy_model = policy_model
        self.belief_model = belief_model
        self.grp_model = grp_model
        self.model_cfg_dict = _asdict(model_cfg)
        self.bel_cfg_dict = _asdict(bel_cfg)
        self.grp_cfg_dict = _asdict(grp_cfg)
        self.grp_params = jax.device_get(grp_params)
        self.bc_params = jax.device_get(bc_params)
        self.bel_params = jax.device_get(bel_params)
        self.belief_t_cal = float(belief_t_cal)

        self.target_updates = int(target_updates)
        self.warmup_updates = int(warmup_updates)
        self.checkpoint_every = int(checkpoint_every)
        self.version_every = int(version_every)
        self.eval_every = int(eval_every)
        self.eval_walls = int(eval_walls)
        self.eval_seed = int(eval_seed)
        self.eval_chunk_walls = int(eval_chunk_walls)
        self.eval_vs_best = bool(eval_vs_best)
        self.degrade_patience = max(1, int(degrade_patience))
        self.early_stop = bool(early_stop)
        self.hf_upload_repo = hf_upload_repo or ""
        self.hf_model_private = bool(hf_model_private)

        if self.checkpoint_every > 0:
            if self.version_every > 0 and (self.version_every % self.checkpoint_every != 0):
                print(
                    f"[VersionedEvalTracker Warning] RL_VERSION_EVERY={self.version_every} is not a multiple of "
                    f"RL_CHECKPOINT_EVERY={self.checkpoint_every}; version saves only trigger on checkpoint boundaries."
                )
            if self.eval_every > 0 and (self.eval_every % self.checkpoint_every != 0):
                print(
                    f"[VersionedEvalTracker Warning] RL_EVAL_EVERY={self.eval_every} is not a multiple of "
                    f"RL_CHECKPOINT_EVERY={self.checkpoint_every}; periodic evals only trigger on checkpoint boundaries."
                )

        self.eval_history_path = self.ckpt_dir / "rl_eval_history.json"
        self.best_ckpt_path = self.ckpt_dir / "rl_best.pkl"
        self.records: List[Dict[str, Any]] = []
        self.evaluated_updates = set()
        self.best_pt = float("-inf")
        self.best_upd: Optional[int] = None
        self.best_pol: Any = None
        self.best_bel: Any = None
        self.best_wall_pts: Optional[List[float]] = None
        self.degrade_streak = 0
        self._evaluator: Optional[CompiledDuplicate1v3Evaluator] = None

        if reset_history:
            for stale in (self.eval_history_path, self.best_ckpt_path):
                if stale.exists():
                    stale.unlink()
        else:
            if self.eval_history_path.exists():
                try:
                    loaded = json.loads(self.eval_history_path.read_text(encoding="utf-8"))
                    if isinstance(loaded, list):
                        self.records = loaded
                        self.evaluated_updates = {int(r["update"]) for r in self.records if "update" in r}
                        if self.records:
                            self.degrade_streak = int(self.records[-1].get("degrade_streak", 0))
                except Exception as exc:
                    print(f"[VersionedEvalTracker Warning] Could not load {self.eval_history_path}: {exc}")

            if self.best_ckpt_path.exists():
                try:
                    with open(self.best_ckpt_path, "rb") as f:
                        db = pickle.load(f)
                    self.best_pol = db.get("policy_params")
                    self.best_bel = db.get("belief_ema_params")
                    self.best_upd = int(db.get("completed_updates", 0)) if db.get("completed_updates") is not None else None
                    ev = db.get("eval_vs_bc") or {}
                    if "tenhou_pt_mean" in ev:
                        self.best_pt = float(ev["tenhou_pt_mean"])
                    self.best_wall_pts = ev.get("wall_mean_pts")
                    print(
                        f"[VersionedEvalTracker] Loaded existing rl_best.pkl "
                        f"(upd={self.best_upd}, pt_vs_bc={self.best_pt:+.2f})"
                    )
                except Exception as exc:
                    print(f"[VersionedEvalTracker Warning] Could not load {self.best_ckpt_path}: {exc}")

    def _get_evaluator(self) -> CompiledDuplicate1v3Evaluator:
        if self._evaluator is None:
            eval_cfg = EvalConfig(
                env_name="red_4p",
                round_mode="half",
                num_walls=self.eval_walls,
                max_steps=2400,
                rank_points=(90.0, 45.0, 0.0, -135.0),
                seed=self.eval_seed,
            )
            self._evaluator = CompiledDuplicate1v3Evaluator(
                policy_model=self.policy_model,
                belief_model=self.belief_model,
                grp_model=self.grp_model,
                grp_base_params=self.grp_params,
                eval_cfg=eval_cfg,
                w_rule=W_PHOENIX_7D,
                opp_rules_known=False,
                belief_t_cal=self.belief_t_cal,
                greedy=True,
                chunk_walls=self.eval_chunk_walls,
            )
        return self._evaluator

    def build_standalone_payload(
        self,
        completed_upd: int,
        policy_params: Any,
        belief_ema_params: Any,
        *,
        eval_vs_bc: Optional[Dict[str, Any]] = None,
        eval_vs_best: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        return jax.device_get(
            {
                "completed_updates": int(completed_upd),
                "model_config": self.model_cfg_dict,
                "belief_config": self.bel_cfg_dict,
                "grp_config": self.grp_cfg_dict,
                "policy_params": policy_params,
                "belief_ema_params": belief_ema_params,
                "grp_base_params": self.grp_params,
                "t_cal": float(self.belief_t_cal),
                "eval_vs_bc": eval_vs_bc,
                "eval_vs_best": eval_vs_best,
            }
        )

    def handle_checkpoint(self, completed_upd: int, saved_resume_path: Path) -> None:
        completed_upd = int(completed_upd)
        need_ver = (self.version_every > 0) and (
            (completed_upd % self.version_every == 0) or (completed_upd >= self.target_updates)
        )
        need_eval = (
            (self.eval_every > 0)
            and ((completed_upd > self.warmup_updates) or (completed_upd >= self.target_updates))
            and ((completed_upd % self.eval_every == 0) or (completed_upd >= self.target_updates))
            and (completed_upd not in self.evaluated_updates)
        )

        eval_vs_bc = None
        eval_vs_best = None
        should_stop = False

        if need_ver or need_eval:
            with open(saved_resume_path, "rb") as f:
                d = pickle.load(f)
            cand_pol = jax.device_get(d.get("params_pi", d.get("policy_params")))
            cand_bel = jax.device_get(d.get("params_b_ema", d.get("belief_ema_params")))
            hist = d.get("history") or []
            train_last = hist[-1] if hist else {}
            del d

            if need_eval:
                evaluator = self._get_evaluator()
                prev_best_pt = self.best_pt
                prev_best_upd = self.best_upd
                prev_best_pol = self.best_pol
                prev_best_bel = self.best_bel

                eval_vs_bc = evaluator.evaluate(cand_pol, cand_bel, self.bc_params, self.bel_params)
                pt_mean = float(eval_vs_bc["tenhou_pt_mean"])
                ci95 = float(eval_vs_bc["tenhou_pt_ci95"])

                paired_diff = None
                paired_ci95 = None
                if self.best_wall_pts is not None and len(self.best_wall_pts) == len(eval_vs_bc.get("wall_mean_pts", [])):
                    diffs = np.asarray(eval_vs_bc["wall_mean_pts"], dtype=np.float64) - np.asarray(
                        self.best_wall_pts, dtype=np.float64
                    )
                    paired_diff = float(diffs.mean())
                    paired_ci95 = (
                        float(1.96 * diffs.std(ddof=1) / np.sqrt(len(diffs))) if len(diffs) > 1 else 0.0
                    )

                if self.eval_vs_best and prev_best_pol is not None and prev_best_upd != completed_upd:
                    eval_vs_best = evaluator.evaluate(cand_pol, cand_bel, prev_best_pol, prev_best_bel)

                is_new_best = (self.best_upd is None) or (pt_mean > prev_best_pt)
                if is_new_best:
                    self.best_pt = pt_mean
                    self.best_upd = completed_upd
                    self.best_pol = cand_pol
                    self.best_bel = cand_bel
                    self.best_wall_pts = list(eval_vs_bc["wall_mean_pts"])
                    self.degrade_streak = 0
                    best_payload = self.build_standalone_payload(
                        completed_upd, cand_pol, cand_bel, eval_vs_bc=eval_vs_bc, eval_vs_best=eval_vs_best
                    )
                    _atomic_dump_pkl(self.best_ckpt_path, best_payload)
                elif pt_mean < (prev_best_pt - max(ci95, 8.0)):
                    self.degrade_streak += 1
                else:
                    self.degrade_streak = 0

                rec = {
                    "update": completed_upd,
                    "eval_seed": self.eval_seed,
                    "eval_walls": self.eval_walls,
                    "total_games": int(eval_vs_bc["total_games"]),
                    "pt_vs_bc": pt_mean,
                    "ci95_vs_bc": ci95,
                    "avg_rank": float(eval_vs_bc["avg_rank"]),
                    "rank_1_rate": float(eval_vs_bc["rank_1_rate"]),
                    "rank_2_rate": float(eval_vs_bc["rank_2_rate"]),
                    "rank_3_rate": float(eval_vs_bc["rank_3_rate"]),
                    "rank_4_rate": float(eval_vs_bc["rank_4_rate"]),
                    "hora_rate": float(eval_vs_bc["hora_rate"]),
                    "deal_in_rate": float(eval_vs_bc["deal_in_rate"]),
                    "riichi_rate": float(eval_vs_bc["riichi_rate"]),
                    "meld_rate": float(eval_vs_bc["meld_rate"]),
                    "eval_wall_s": float(eval_vs_bc["eval_wall_s"]),
                    "is_best": bool(is_new_best),
                    "best_pt_so_far": float(self.best_pt),
                    "best_upd_so_far": int(self.best_upd) if self.best_upd is not None else completed_upd,
                    "paired_diff_vs_best": paired_diff,
                    "paired_ci95_vs_best": paired_ci95,
                    "degrade_streak": int(self.degrade_streak),
                    "wall_mean_pts": eval_vs_bc["wall_mean_pts"],
                    "train_ora": float(train_last["oracle_loss"]) if "oracle_loss" in train_last else None,
                    "train_bel": float(train_last["belief_loss"]) if "belief_loss" in train_last else None,
                    "train_ent": float(train_last["policy_entropy"]) if "policy_entropy" in train_last else None,
                    "train_kl": float(train_last["approx_kl"]) if "approx_kl" in train_last else None,
                    "train_beta_r": float(train_last["beta_risk"]) if "beta_risk" in train_last else None,
                    "train_beta_o": float(train_last["beta_offense"]) if "beta_offense" in train_last else None,
                    "adv_rms": float(train_last["adv_rms"]) if "adv_rms" in train_last else None,
                }
                if eval_vs_best is not None:
                    rec["eval_vs_prev_best"] = {
                        "target_upd": prev_best_upd,
                        "pt_mean": float(eval_vs_best["tenhou_pt_mean"]),
                        "ci95": float(eval_vs_best["tenhou_pt_ci95"]),
                        "avg_rank": float(eval_vs_best["avg_rank"]),
                    }
                self.records.append(rec)
                self.evaluated_updates.add(completed_upd)
                _atomic_write_json(self.eval_history_path, self.records)

                tag_best = " ★ NEW BEST!" if is_new_best else (
                    f" [DEGRADE {self.degrade_streak}/{self.degrade_patience}]"
                    if self.degrade_streak > 0
                    else f" (best@{self.best_upd}={self.best_pt:+.2f})"
                )
                paired_str = (
                    f" | Δbest={paired_diff:+.2f}±{paired_ci95:.2f}"
                    if (not is_new_best and paired_diff is not None)
                    else ""
                )
                print(
                    f"[RL EVAL upd={completed_upd:04d}] vs 3x BC ({int(eval_vs_bc['total_games'])}g, "
                    f"{eval_vs_bc['eval_wall_s']:.1f}s) | pt={pt_mean:+.2f}±{ci95:.2f} | "
                    f"rank={eval_vs_bc['avg_rank']:.3f} | "
                    f"1st={eval_vs_bc['rank_1_rate']*100:.1f}% 4th={eval_vs_bc['rank_4_rate']*100:.1f}% | "
                    f"hora={eval_vs_bc['hora_rate']*100:.1f}% deal={eval_vs_bc['deal_in_rate']*100:.1f}% "
                    f"riichi={eval_vs_bc['riichi_rate']*100:.1f}% meld={eval_vs_bc['meld_rate']*100:.1f}%"
                    f"{paired_str}{tag_best}",
                    flush=True,
                )
                if eval_vs_best is not None:
                    print(
                        f"  └─ vs rl_best(upd={prev_best_upd}) | "
                        f"pt={eval_vs_best['tenhou_pt_mean']:+.2f}±{eval_vs_best['tenhou_pt_ci95']:.2f} | "
                        f"rank={eval_vs_best['avg_rank']:.3f}",
                        flush=True,
                    )
                should_stop = self.early_stop and (self.degrade_streak >= self.degrade_patience)

            if need_ver:
                ver_path = self.ckpt_dir / f"rl_upd_{completed_upd:04d}.pkl"
                ver_payload = self.build_standalone_payload(
                    completed_upd, cand_pol, cand_bel, eval_vs_bc=eval_vs_bc, eval_vs_best=eval_vs_best
                )
                _atomic_dump_pkl(ver_path, ver_payload)
                print(f"[RL Version] Saved {ver_path.name} ({ver_path.stat().st_size / 1e6:.2f} MB)", flush=True)

            gc.collect()

        if self.hf_upload_repo:
            try:
                from mahjong_rl.data.hf_sync import upload_checkpoints_to_hf

                upload_checkpoints_to_hf(self.ckpt_dir, self.hf_upload_repo, private=self.hf_model_private)
            except Exception as exc:
                print(f"[HF Backup Warning] upd={completed_upd}: {exc}", flush=True)

        if should_stop:
            raise _EarlyStopRL(
                f"Early stopping RL at upd={completed_upd}: {self.degrade_streak} consecutive evals "
                f"below rl_best (upd={self.best_upd}, pt={self.best_pt:+.2f})."
            )


def plot_eval_progress_curves(records: List[Dict[str, Any]], save_path: Optional[Path] = None) -> None:
    """Plot 2x2 Duplicate 1v3 progress dashboard (pt, rank/1st/4th, playstyle rates, entropy vs pt)."""
    if not records:
        print("[Eval Plot] No eval records available yet.")
        return

    import matplotlib.pyplot as plt

    recs = sorted(records, key=lambda r: int(r["update"]))
    xs = np.array([int(r["update"]) for r in recs], dtype=np.int32)
    pts = np.array([float(r["pt_vs_bc"]) for r in recs], dtype=np.float64)
    cis = np.array([float(r.get("ci95_vs_bc", 0.0)) for r in recs], dtype=np.float64)
    ranks = np.array([float(r["avg_rank"]) for r in recs], dtype=np.float64)
    r1 = np.array([float(r["rank_1_rate"]) * 100.0 for r in recs], dtype=np.float64)
    r4 = np.array([float(r["rank_4_rate"]) * 100.0 for r in recs], dtype=np.float64)
    hora = np.array([float(r["hora_rate"]) * 100.0 for r in recs], dtype=np.float64)
    deal = np.array([float(r["deal_in_rate"]) * 100.0 for r in recs], dtype=np.float64)
    riichi = np.array([float(r["riichi_rate"]) * 100.0 for r in recs], dtype=np.float64)
    meld = np.array([float(r["meld_rate"]) * 100.0 for r in recs], dtype=np.float64)

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))

    # 1. Tenhou pt vs 3x frozen BC with 95% CI band
    ax = axes[0, 0]
    ax.plot(xs, pts, marker="o", linewidth=2, color="#1f77b4", label="Tenhou pt vs 3x BC")
    ax.fill_between(xs, pts - cis, pts + cis, color="#1f77b4", alpha=0.18, label="95% CI")
    ax.axhline(0.0, color="#666666", linestyle="--", linewidth=1, label="BC Baseline (0 pt)")
    best_i = int(np.argmax(pts))
    ax.scatter([xs[best_i]], [pts[best_i]], marker="*", s=220, color="#d62728", zorder=5, label=f"Best @upd {xs[best_i]} ({pts[best_i]:+.1f})")
    ax.set_title("Strength Curve: Tenhou pt vs 3x Frozen BC")
    ax.set_xlabel("Update")
    ax.set_ylabel("Paired Tenhou pt / game")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)

    # 2. Average rank + 1st% / 4th% rates
    ax = axes[0, 1]
    l1 = ax.plot(xs, ranks, marker="s", linewidth=2, color="#2ca02c", label="Avg Rank (left)")
    ax.axhline(2.5, color="#888888", linestyle="--", linewidth=1)
    ax.set_title("Avg Rank & 1st / 4th Rate")
    ax.set_xlabel("Update")
    ax.set_ylabel("Average Rank (lower=better)", color="#2ca02c")
    ax.grid(True, alpha=0.3)
    ax_r = ax.twinx()
    l2 = ax_r.plot(xs, r1, marker="^", linestyle="-.", linewidth=1.6, color="#ff7f0e", label="1st % (right)")
    l3 = ax_r.plot(xs, r4, marker="v", linestyle=":", linewidth=1.6, color="#d62728", label="4th % (right)")
    ax_r.set_ylabel("Rate (%)")
    lines = l1 + l2 + l3
    ax.legend(lines, [l.get_label() for l in lines], loc="best", fontsize=8)

    # 3. Playstyle telemetry
    ax = axes[1, 0]
    ax.plot(xs, hora, marker="o", linewidth=1.7, label="Hora %")
    ax.plot(xs, deal, marker="o", linewidth=1.7, label="Deal-in %")
    ax.plot(xs, riichi, marker="o", linewidth=1.7, label="Riichi %")
    ax.plot(xs, meld, marker="o", linewidth=1.7, label="Meld %")
    ax.set_title("Playstyle Telemetry (Per-Hand Rates)")
    ax.set_xlabel("Update")
    ax.set_ylabel("Rate (%)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)

    # 4. Policy Entropy vs Tenhou pt
    ax = axes[1, 1]
    ents = [r.get("train_ent") for r in recs]
    if all(e is not None for e in ents):
        ent_arr = np.array(ents, dtype=np.float64)
        e1 = ax.plot(xs, ent_arr, marker="o", linewidth=1.8, color="#9467bd", label="Policy Entropy (left)")
        ax.axhline(0.35, color="#9467bd", linestyle=":", linewidth=1, label="Collapse Floor (0.35)")
        ax.set_ylabel("Policy Entropy", color="#9467bd")
        ax2 = ax.twinx()
        e2 = ax2.plot(xs, pts, marker="s", linestyle="--", linewidth=1.6, color="#1f77b4", label="Tenhou pt (right)")
        ax2.set_ylabel("Tenhou pt vs BC", color="#1f77b4")
        el = e1 + e2
        ax.legend(el, [l.get_label() for l in el], loc="best", fontsize=8)
        ax.set_title("Policy Entropy vs Tenhou pt")
    else:
        ax.plot(xs, r1 - r4, marker="o", linewidth=1.8, color="#9467bd", label="1st% - 4th% Spread")
        ax.axhline(0.0, color="#666666", linestyle="--", linewidth=1)
        ax.set_ylabel("Percentage Points")
        ax.set_title("Top-Bottom Placement Spread (1st% - 4th%)")
        ax.legend(loc="best", fontsize=8)
    ax.set_xlabel("Update")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=140)
    plt.show()
    plt.close(fig)

## 8. Phase 1/2 (`train_rl_v6`) + Phase 3 (`distill_sdtap`) 行为空间自蒸馏

In [ ]:
import matplotlib.pyplot as plt
from mahjong_rl.config import OracleConfig, RLConfig, RNaDConfig
from mahjong_rl.trainers.distill_sdtap import distill_sdtap
from mahjong_rl.trainers.train_rl import train_rl_v6

rl_res = None
rl_hist = []
_loaded_outer_snaps = []
final_rl_params = None
final_rl_bel_params = None
best_rl_params = None
best_rl_bel_params = None
best_rl_meta = None
sdtap_params = None
deploy_params = bc_params
deploy_bel_params = bel_params

_ckpt_src = Path(CFG["PREEXISTING_CKPT_DIR"]) if CFG["PREEXISTING_CKPT_DIR"] else CKPT_DIR
rl_resume_path = _ckpt_src / "rl_resume_state.pkl"
rl_ckpt_path = _ckpt_src / "rl_phase2.pkl"
rl_resume_dst = CKPT_DIR / "rl_resume_state.pkl"

# Determine completed RL updates from existing resumable checkpoint (if any)
_existing_rl_file = rl_resume_path if rl_resume_path.exists() else (rl_ckpt_path if rl_ckpt_path.exists() else None)
_completed_rl_updates = 0
if _existing_rl_file is not None and not CFG.get("FORCE_RETRAIN_RL", False):
    with open(_existing_rl_file, "rb") as f:
        _meta_rl = pickle.load(f)
    if "completed_updates" in _meta_rl:
        _completed_rl_updates = int(_meta_rl["completed_updates"])
    elif "history" in _meta_rl and _meta_rl["history"]:
        _completed_rl_updates = int(_meta_rl["history"][-1].get("update", -1)) + 1

_need_rl_train = CFG["RUN_PHASE12_RL"] and (
    CFG.get("FORCE_RETRAIN_RL", False)
    or _existing_rl_file is None
    or (CFG.get("RL_RESUME", True) and _completed_rl_updates < int(CFG["RL_NUM_UPDATES"]))
)

if _need_rl_train:
    if CFG.get("FORCE_RETRAIN_RL", False):
        for _stale in [
            CKPT_DIR / "rl_resume_state.pkl",
            CKPT_DIR / "rl_phase2.pkl",
            CKPT_DIR / "rl_best.pkl",
            CKPT_DIR / "deploy_sdtap.pkl",
            CKPT_DIR / "rl_eval_history.json",
            CKPT_DIR / "rl_curves.png",
            CKPT_DIR / "rl_eval_curves.png",
            *CKPT_DIR.glob("rl_upd_*.pkl"),
        ]:
            if _stale.exists():
                _stale.unlink()
        if CFG.get("HF_UPLOAD_REPO"):
            try:
                from mahjong_rl.data.hf_sync import upload_checkpoints_to_hf
                upload_checkpoints_to_hf(
                    CKPT_DIR,
                    CFG["HF_UPLOAD_REPO"],
                    private=bool(CFG.get("HF_MODEL_PRIVATE", True)),
                    delete_patterns=["rl_*.pkl", "deploy_sdtap.pkl", "rl_eval_history.json", "rl_curves.png", "rl_eval_curves.png"],
                    commit_message="Reset stale RL checkpoints before clean retrain",
                )
            except Exception as _exc:
                print(f"[HF Stale-RL Cleanup Warning] {type(_exc).__name__}: {_exc}")
    t0 = time.perf_counter()
    rl_cfg = RLConfig(
        env_name="red_4p",
        round_mode="half",
        model=model_cfg,
        belief=bel_cfg,
        grp=grp_cfg,
        oracle=OracleConfig(replay_updates=int(CFG.get("RL_ORACLE_REPLAY_UPDATES", 6))),
        rnad=RNaDConfig(
            warmup_updates=CFG["RL_WARMUP_UPDATES"],
            outer_reset_steps=CFG["RL_OUTER_RESET_STEPS"],
            pfsp_snapshot_interval=int(CFG.get("RL_PFSP_SNAPSHOT_INTERVAL", 200)),
            reset_warmup_steps=30,
            lambda_intra=float(CFG.get("RL_LAMBDA_INTRA", 1.0)),
            lambda_inter=float(CFG.get("RL_LAMBDA_INTER", 0.0)),
            mask_unterminated_tail=bool(CFG.get("RL_MASK_UNTERMINATED_TAIL", True)),
            freeze_belief=bool(CFG.get("RL_FREEZE_BELIEF", True)),
            policy_lr=float(CFG.get("RL_POLICY_LR", 3e-5)),
            alpha_rnad=float(CFG.get("RL_ALPHA_RNAD", 0.12)),
            rho_mix_bc=float(CFG.get("RL_RHO_MIX_BC", 0.10)),
            rho_mix_uniform=float(CFG.get("RL_RHO_MIX_UNIFORM", 0.005)),
            entropy_coef=float(CFG.get("RL_ENTROPY_COEF", 0.002)),
            phoenix_7d_only=bool(CFG.get("RL_PHOENIX_7D_ONLY", True)),
            pfsp_frac=float(CFG.get("RL_PFSP_FRAC", 0.625)),
            opp_temperature=float(CFG.get("RL_OPP_TEMPERATURE", 0.35)),
            ron_k_max=max(8192, 2 * CFG["RL_NUM_ENVS"] * CFG["RL_NUM_STEPS"] // max(N_DEVICES, 1)),
            audit_interval=25,
        ),
        num_envs=CFG["RL_NUM_ENVS"],
        num_steps=CFG["RL_NUM_STEPS"],
        num_updates=CFG["RL_NUM_UPDATES"],
        update_epochs=CFG["RL_UPDATE_EPOCHS"],
        minibatch_size=CFG["RL_MINIBATCH_SIZE"],
        seed=CFG["SEED"],
    )

    tracker = VersionedEvalTracker(
        ckpt_dir=CKPT_DIR,
        policy_model=pol_model,
        belief_model=bel_model,
        grp_model=grp_model,
        model_cfg=model_cfg,
        bel_cfg=bel_cfg,
        grp_cfg=grp_cfg,
        grp_params=grp_params,
        bc_params=bc_params,
        bel_params=bel_params,
        belief_t_cal=belief_t_cal,
        target_updates=int(CFG["RL_NUM_UPDATES"]),
        warmup_updates=int(CFG["RL_WARMUP_UPDATES"]),
        checkpoint_every=int(CFG.get("RL_CHECKPOINT_EVERY", 50)),
        version_every=int(CFG.get("RL_VERSION_EVERY", 100)),
        eval_every=int(CFG.get("RL_EVAL_EVERY", 100)),
        eval_walls=int(CFG.get("RL_EVAL_WALLS", 32)),
        eval_seed=int(CFG.get("RL_EVAL_SEED", 2026)),
        eval_chunk_walls=int(CFG.get("RL_EVAL_CHUNK_WALLS", 8)),
        eval_vs_best=bool(CFG.get("RL_EVAL_VS_BEST", False)),
        degrade_patience=int(CFG.get("RL_DEGRADE_PATIENCE", 3)),
        early_stop=bool(CFG.get("RL_EARLY_STOP", False)),
        hf_upload_repo=CFG.get("HF_UPLOAD_REPO", ""),
        hf_model_private=bool(CFG.get("HF_MODEL_PRIVATE", True)),
        reset_history=bool(CFG.get("FORCE_RETRAIN_RL", False)),
    )

    _active_resume = (
        _existing_rl_file
        if (CFG.get("RL_RESUME", True) and not CFG.get("FORCE_RETRAIN_RL", False) and _existing_rl_file is not None)
        else None
    )
    _early_stopped = False
    try:
        rl_res = train_rl_v6(
            rl_cfg,
            policy_params_init=bc_params,
            belief_params_init=bel_params,
            belief_t_cal=belief_t_cal,
            grp_params_init=grp_params,
            enable_self_healing=True,
            resume_path=_active_resume,
            checkpoint_path=rl_resume_dst,
            checkpoint_every=CFG.get("RL_CHECKPOINT_EVERY", 50),
            on_checkpoint=tracker.handle_checkpoint,
        )
    except _EarlyStopRL as exc:
        _early_stopped = True
        print(f"\n[Phase 1/2 RL EARLY-STOPPED] {exc}")

    if rl_res is not None:
        print(
            f"Phase 1/2 RL finished in {(time.perf_counter() - t0)/60:.1f} min | "
            f"total_updates={len(rl_res.history)} | snapshots={len(rl_res.outer_snapshots)}"
        )
        rl_hist = list(rl_res.history)
        _loaded_outer_snaps = list(rl_res.outer_snapshots)
        final_rl_params = rl_res.policy_params
        final_rl_bel_params = rl_res.belief_ema_params
        _atomic_dump_pkl(
            CKPT_DIR / "rl_phase2.pkl",
            {
                "completed_updates": int(CFG["RL_NUM_UPDATES"]),
                "policy_params": rl_res.policy_params,
                "belief_ema_params": rl_res.belief_ema_params,
                "oracle_params": rl_res.oracle_params,
                "t_cal": float(belief_t_cal),
                "history": rl_res.history,
            },
        )
    elif _early_stopped and rl_resume_dst.exists():
        with open(rl_resume_dst, "rb") as f:
            _d_es = pickle.load(f)
        rl_hist = list(_d_es.get("history") or [])
        _loaded_outer_snaps = list(_d_es.get("outer_snapshots") or [])
        final_rl_params = jax.device_put(_d_es.get("params_pi", _d_es.get("policy_params")))
        final_rl_bel_params = jax.device_put(_d_es.get("params_b_ema", _d_es.get("belief_ema_params")))
        _es_upd = int(_d_es.get("completed_updates", len(rl_hist)))
        _atomic_dump_pkl(
            CKPT_DIR / "rl_phase2.pkl",
            {
                "completed_updates": _es_upd,
                "policy_params": _d_es.get("params_pi", _d_es.get("policy_params")),
                "belief_ema_params": _d_es.get("params_b_ema", _d_es.get("belief_ema_params")),
                "oracle_params": _d_es.get("params_oracle", _d_es.get("oracle_params")),
                "t_cal": float(belief_t_cal),
                "history": rl_hist,
            },
        )

    if rl_hist:
        xs = [r["update"] for r in rl_hist]
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].plot(xs, [r["oracle_loss"] for r in rl_hist], label="Oracle Loss")
        axes[0].plot(xs, [r["belief_loss"] for r in rl_hist], label="Belief Loss")
        axes[0].set_title("Oracle & Belief Convergence"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(xs, [r["policy_entropy"] for r in rl_hist], label="Policy Entropy")
        axes[1].plot(xs, [r["approx_kl"] * 10.0 for r in rl_hist], label="10x Approx KL")
        axes[1].set_title("Policy Stability"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(xs, [r["beta_risk"] for r in rl_hist], label="beta_risk")
        axes[2].plot(xs, [r["beta_offense"] for r in rl_hist], label="beta_offense")
        axes[2].set_title("Disjoint Bounded Physics Gates"); axes[2].legend(); axes[2].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(CKPT_DIR / "rl_curves.png", dpi=140)
        plt.show()
        plt.close(fig)

    if tracker.records:
        plot_eval_progress_curves(tracker.records, save_path=CKPT_DIR / "rl_eval_curves.png")
elif CFG["RUN_PHASE12_RL"] and _existing_rl_file is not None:
    print(
        f"[Phase 1/2 RL] Skipping training — already at {_completed_rl_updates}/{CFG['RL_NUM_UPDATES']} updates "
        f"({_existing_rl_file}); increase RL_NUM_UPDATES to continue or set FORCE_RETRAIN_RL=True"
    )
    with open(_existing_rl_file, "rb") as f:
        d_rl = pickle.load(f)
    final_rl_params = jax.device_put(d_rl.get("params_pi", d_rl.get("policy_params")))
    final_rl_bel_params = jax.device_put(d_rl.get("params_b_ema", d_rl.get("belief_ema_params")))
    _loaded_outer_snaps = list(d_rl.get("outer_snapshots") or [])
    _hist_file = (
        CKPT_DIR / "rl_eval_history.json"
        if (CKPT_DIR / "rl_eval_history.json").exists()
        else (_ckpt_src / "rl_eval_history.json" if (_ckpt_src / "rl_eval_history.json").exists() else None)
    )
    if _hist_file is not None:
        try:
            _loaded_eval_recs = json.loads(_hist_file.read_text(encoding="utf-8"))
            plot_eval_progress_curves(_loaded_eval_recs, save_path=CKPT_DIR / "rl_eval_curves.png")
        except Exception as exc:
            print(f"[Eval Plot Warning] Could not plot {_hist_file}: {exc}")

if CFG["RUN_PHASE12_RL"]:
    _allow_src_best = (not CFG.get("FORCE_RETRAIN_RL", False)) and (_ckpt_src != CKPT_DIR)
    _best_file = (
        CKPT_DIR / "rl_best.pkl"
        if (CKPT_DIR / "rl_best.pkl").exists()
        else (_ckpt_src / "rl_best.pkl" if (_allow_src_best and (_ckpt_src / "rl_best.pkl").exists()) else None)
    )
    if _best_file is not None:
        with open(_best_file, "rb") as f:
            _d_best = pickle.load(f)
        best_rl_params = jax.device_put(_d_best["policy_params"])
        best_rl_bel_params = jax.device_put(_d_best["belief_ema_params"])
        best_rl_meta = {
            "completed_updates": _d_best.get("completed_updates"),
            "eval_vs_bc": _d_best.get("eval_vs_bc"),
        }

    if CFG.get("RL_USE_BEST_FOR_DEPLOY", True) and best_rl_params is not None:
        deploy_params = best_rl_params
        deploy_bel_params = best_rl_bel_params
        _b_ev = (best_rl_meta or {}).get("eval_vs_bc") or {}
        _b_pt = _b_ev.get("tenhou_pt_mean")
        _b_pt_s = f"{_b_pt:+.2f}" if _b_pt is not None else "n/a"
        print(
            f"[RL Deploy] Selected rl_best.pkl (upd={best_rl_meta['completed_updates']}, "
            f"val pt vs BC={_b_pt_s})"
        )
    elif final_rl_params is not None:
        deploy_params = final_rl_params
        deploy_bel_params = final_rl_bel_params

if CFG["RUN_PHASE3_SDTAP"] and len(_loaded_outer_snaps) >= 2 and (val_shards or train_shards):
    t0 = time.perf_counter()
    # Distill last up to 5 outer R-NaD snapshots in behavioral probability space
    snaps_to_distill = _loaded_outer_snaps[-5:]
    _, sdtap_params, sdtap_hist = distill_sdtap(
        val_shards + train_shards[:2],
        snaps_to_distill,
        belief_model=bel_model,
        belief_ema_params=deploy_bel_params,
        grp_model=grp_model,
        grp_base_params=grp_params,
        model_cfg=model_cfg,
        belief_t_cal=belief_t_cal,
        batch_size=CFG["SDTAP_BATCH_SIZE"],
        num_epochs=CFG["SDTAP_EPOCHS"],
        seed=CFG["SEED"],
    )
    print(f"Phase 3 SD-TAP finished in {time.perf_counter() - t0:.1f}s | hist={sdtap_hist}")
    if not (CFG.get("RL_USE_BEST_FOR_DEPLOY", True) and best_rl_params is not None):
        deploy_params = sdtap_params
    _atomic_dump_pkl(
        CKPT_DIR / "deploy_sdtap.pkl",
        {
            "model_config": asdict(model_cfg),
            "belief_config": asdict(bel_cfg),
            "grp_config": asdict(grp_cfg),
            "policy_params": sdtap_params,
            "belief_ema_params": deploy_bel_params,
            "grp_base_params": grp_params,
            "t_cal": float(belief_t_cal),
            "sdtap_history": sdtap_hist,
        },
    )
elif (not _need_rl_train) and CFG["RUN_PHASE3_SDTAP"] and (CKPT_DIR / "deploy_sdtap.pkl").exists():
    with open(CKPT_DIR / "deploy_sdtap.pkl", "rb") as f:
        _d_sdtap = pickle.load(f)
    sdtap_params = jax.device_put(_d_sdtap["policy_params"])

## 9. 严格固定牌山全半庄 Duplicate 1v3 评测 & 零样本多规则矩阵

In [ ]:
import mahjax
from mahjong_rl.algo.utility import W_M_LEAGUE, W_PHOENIX_7D, W_PHOENIX_10D
from mahjong_rl.config import EvalConfig
from mahjong_rl.eval.duplicate_eval import evaluate_duplicate_1v3
from mahjong_rl.eval.mjai_bridge import make_v6_policy_actor

_eval_history_records = []
_eval_hist_path = CKPT_DIR / "rl_eval_history.json"
if _eval_hist_path.exists():
    try:
        _eval_history_records = json.loads(_eval_hist_path.read_text(encoding="utf-8"))
    except Exception:
        pass

full_report = {
    "cfg": CFG,
    "phase0b_grp_val": grp_val,
    "phase0a_belief_val": bel_val,
    "phase0c_bc_val": bc_val,
    "rl_eval_history": _eval_history_records,
    "duplicate_1v3_eval": {},
}

if CFG["RUN_DUPLICATE_EVAL"]:
    _test_seed = int(CFG.get("EVAL_TEST_SEED", 9999))
    eval_cfg = EvalConfig(
        env_name="red_4p",
        round_mode="half",
        num_walls=CFG["EVAL_WALLS"],
        max_steps=2400,
        rank_points=(90.0, 45.0, 0.0, -135.0),
        seed=_test_seed,
    )

    candidates = [("phase0c_bc_7d", bc_params, bel_params, W_PHOENIX_7D)]
    if CFG["RUN_PHASE12_RL"]:
        candidates.extend([
            ("v6_final_7d", deploy_params, deploy_bel_params, W_PHOENIX_7D),
            ("v6_final_10d_zeroshot", deploy_params, deploy_bel_params, W_PHOENIX_10D),
            ("v6_final_mleague_zeroshot", deploy_params, deploy_bel_params, W_M_LEAGUE),
        ])
        if best_rl_params is not None and best_rl_params is not deploy_params:
            candidates.append(("rl_best_7d", best_rl_params, best_rl_bel_params, W_PHOENIX_7D))
    else:
        candidates.extend([
            ("phase0c_bc_10d_zeroshot", bc_params, bel_params, W_PHOENIX_10D),
            ("phase0c_bc_mleague_zeroshot", bc_params, bel_params, W_M_LEAGUE),
        ])

    print(
        f"=== Independent Test Duplicate 1v3 ({4 * CFG['EVAL_WALLS']} games/entry vs rule_based_player | seed={_test_seed}) ==="
    )
    for tag, p_pol, p_bel, w_rule in candidates:
        actor = make_v6_policy_actor(
            policy_model=pol_model,
            policy_params=p_pol,
            belief_model=bel_model,
            belief_ema_params=p_bel,
            grp_model=grp_model,
            grp_base_params=grp_params,
            w_rule=w_rule,
            opp_rules_known=False,
            belief_t_cal=belief_t_cal,
            greedy=True,
        )
        res = evaluate_duplicate_1v3(
            actor, mahjax.rule_based_player, eval_cfg, chunk_walls=CFG["EVAL_CHUNK_WALLS"]
        )
        full_report["duplicate_1v3_eval"][tag] = res
        print(
            f"  [{tag:26s}] pt={res['tenhou_pt_mean']:+.2f} ± {res['tenhou_pt_ci95']:.2f} | "
            f"rank={res['avg_rank']:.3f} | 1st={res['rank_1_rate']*100:.1f}% 4th={res['rank_4_rate']*100:.1f}% | "
            f"hora={res['hora_rate']*100:.1f}% deal_in={res['deal_in_rate']*100:.1f}% "
            f"riichi={res['riichi_rate']*100:.1f}% meld={res['meld_rate']*100:.1f}%"
        )

    # Zero-recompilation 1v3 head-to-head on independent test walls: all RL variants vs 3x phase0c_bc_7d
    if CFG["RUN_PHASE12_RL"]:
        h2h_entries = []
        if final_rl_params is not None:
            h2h_entries.append(("rl_final_7d_vs_bc_7d", final_rl_params, final_rl_bel_params))
        if best_rl_params is not None:
            h2h_entries.append(("rl_best_7d_vs_bc_7d", best_rl_params, best_rl_bel_params))
        if sdtap_params is not None:
            h2h_entries.append(("v6_sdtap_7d_vs_bc_7d", sdtap_params, deploy_bel_params))
        if not h2h_entries:
            h2h_entries.append(("v6_deploy_7d_vs_bc_7d", deploy_params, deploy_bel_params))

        print(
            f"\n=== Independent Head-to-Head Duplicate 1v3 ({4 * CFG['EVAL_WALLS']} games/entry vs 3x phase0c_bc_7d | seed={_test_seed}) ==="
        )
        h2h_eval = CompiledDuplicate1v3Evaluator(
            policy_model=pol_model,
            belief_model=bel_model,
            grp_model=grp_model,
            grp_base_params=grp_params,
            eval_cfg=eval_cfg,
            w_rule=W_PHOENIX_7D,
            opp_rules_known=False,
            belief_t_cal=belief_t_cal,
            greedy=True,
            chunk_walls=CFG["EVAL_CHUNK_WALLS"],
        )
        h2h_summary = []
        for tag, p_pol, p_bel in h2h_entries:
            res_h2h = h2h_eval.evaluate(p_pol, p_bel, bc_params, bel_params)
            full_report["duplicate_1v3_eval"][tag] = res_h2h
            h2h_summary.append((tag, res_h2h["tenhou_pt_mean"], res_h2h["tenhou_pt_ci95"]))
            print(
                f"  [{tag:26s}] pt={res_h2h['tenhou_pt_mean']:+.2f} ± {res_h2h['tenhou_pt_ci95']:.2f} | "
                f"rank={res_h2h['avg_rank']:.3f} | 1st={res_h2h['rank_1_rate']*100:.1f}% 4th={res_h2h['rank_4_rate']*100:.1f}% | "
                f"hora={res_h2h['hora_rate']*100:.1f}% deal_in={res_h2h['deal_in_rate']*100:.1f}% "
                f"riichi={res_h2h['riichi_rate']*100:.1f}% meld={res_h2h['meld_rate']*100:.1f}% "
                f"({res_h2h['eval_wall_s']:.1f}s)"
            )
        best_tag, best_pt_test, best_ci_test = max(h2h_summary, key=lambda x: x[1])
        print(f"  => Independent Test Winner vs 3x BC: {best_tag} ({best_pt_test:+.2f} ± {best_ci_test:.2f} pt)")

report_path = CKPT_DIR / "full_report.json"
report_path.write_text(json.dumps(full_report, indent=2), encoding="utf-8")
print(f"\nSaved full audit report to {report_path}")

## 10. 可选：自动备份权重与报表至 Hugging Face Hub

In [ ]:
from mahjong_rl.data.hf_sync import upload_checkpoints_to_hf

if CFG.get("HF_UPLOAD_REPO"):
    upload_checkpoints_to_hf(
        CKPT_DIR,
        CFG["HF_UPLOAD_REPO"],
        private=CFG.get("HF_MODEL_PRIVATE", True),
    )
else:
    print("HF_UPLOAD_REPO is empty; all checkpoints and reports remain persisted under:", CKPT_DIR)

---
## 11. 常用工具箱（独立小单元，按需单独运行，不干扰主流程 Run All）

以下每个单元格都自带依赖和路径兜底，可在任意时刻单独点击运行（例如：校验 HF Token、查看双卡显存与 20 GB 磁盘余额、**中途断点上传/拉取 `.npz` 分片**、双向同步权重、体检分片完整性、热更新代码免重启内核）。

> 涉及上传/下载/清理的工具单元默认 `RUN_TOOL = False`，保证点击 **Run All** 时安全跳过；单独复用时将其改为 `True` 运行即可。

### 工具 11.1：HF 账号绑定自检 + GPU/显存/磁盘看板 + 分片状态体检（只读安全）

In [ ]:
from pathlib import Path
from mahjong_rl.data.hf_sync import inspect_kaggle_resources, inspect_v6_shards, resolve_hf_token, verify_hf_auth

_work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
_tmp = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else (_work / "tmp")
_shard_dir = globals().get("SHARD_DIR", _work / "v6_shards_2026")

if resolve_hf_token():
    verify_hf_auth()
else:
    print("[HF Auth] 未检测到 HF_TOKEN（如需同步 HF，请在 Kaggle 顶部 Add-ons -> Secrets 添加并勾选 HF_TOKEN）")

print()
inspect_kaggle_resources(_work, _tmp)
print()
inspect_v6_shards(_shard_dir)

### 工具 11.2：一键上传 `.npz` 分片到 HF Dataset (`npz/`) + 自动迁移根目录旧分片

In [ ]:
RUN_TOOL = False      # 单独运行时改成 True
MIGRATE_ONLY = False  # True=仅将 HF 仓库根目录的旧分片服务端零流量迁移到 npz/，不上传本地新分片
HF_DATASET_REPO = globals().get("CFG", {}).get("HF_DATASET_REPO", "") or "myouo/tenhou-2026-v6-shards"
HF_DATASET_SUBDIR = globals().get("CFG", {}).get("HF_DATASET_SUBDIR", "npz")
PRIVATE = True

if RUN_TOOL:
    from pathlib import Path
    from mahjong_rl.data.hf_sync import migrate_hf_root_shards_to_subdir, upload_v6_shards_to_hf

    if MIGRATE_ONLY:
        migrate_hf_root_shards_to_subdir(HF_DATASET_REPO, path_in_repo=HF_DATASET_SUBDIR)
    else:
        _work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
        _shard_dir = globals().get("SHARD_DIR", _work / "v6_shards_2026")
        upload_v6_shards_to_hf(
            _shard_dir,
            HF_DATASET_REPO,
            path_in_repo=HF_DATASET_SUBDIR,
            private=PRIVATE,
            migrate_root=True,
        )
else:
    print("Skipped Tool 11.2 (set RUN_TOOL = True to upload shards to npz/ and auto-migrate legacy root files).")

### 工具 11.3：一键从 Hugging Face Dataset (`npz/`) 秒拉 `.npz` 分片（开新会话跳过转换 / 接力续转）

In [ ]:
RUN_TOOL = False  # 单独运行时改成 True
HF_DATASET_REPO = globals().get("CFG", {}).get("HF_DATASET_REPO", "") or "myouo/tenhou-2026-v6-shards"
HF_DATASET_SUBDIR = globals().get("CFG", {}).get("HF_DATASET_SUBDIR", "npz")
MAX_SHARDS = None  # None=全量拉取；也可填整数（如 30）只拉前 30 个分片快速实验

if RUN_TOOL:
    from pathlib import Path
    from mahjong_rl.data.hf_sync import download_v6_shards_from_hf

    _work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    _shard_dir = globals().get("SHARD_DIR", _work / "v6_shards_2026")
    download_v6_shards_from_hf(HF_DATASET_REPO, _shard_dir, path_in_repo=HF_DATASET_SUBDIR, max_shards=MAX_SHARDS)
else:
    print("Skipped Tool 11.3 (set RUN_TOOL = True and fill HF_DATASET_REPO to download shards from npz/).")

### 工具 11.4：一键双向同步模型权重（`checkpoints` 上传 / 拉取，方便跨会话分段训练）

In [ ]:
RUN_TOOL = False      # 单独运行时改成 True
ACTION = "upload"     # "upload"（备份权重到 HF）或 "download"（从 HF 拉回权重续训）
HF_MODEL_REPO = globals().get("CFG", {}).get("HF_UPLOAD_REPO", "") or "myouo/sovereign-mahjong-v6"
PRIVATE = True

if RUN_TOOL:
    from pathlib import Path
    from mahjong_rl.data.hf_sync import download_checkpoints_from_hf, upload_checkpoints_to_hf

    _work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    _ckpt_dir = globals().get("CKPT_DIR", _work / "checkpoints" / "sovereign_v6")
    if ACTION == "upload":
        upload_checkpoints_to_hf(_ckpt_dir, HF_MODEL_REPO, private=PRIVATE)
    elif ACTION == "download":
        download_checkpoints_from_hf(HF_MODEL_REPO, _ckpt_dir)
    else:
        raise ValueError(f"Unknown ACTION={ACTION!r}; choose 'upload' or 'download'")
else:
    print("Skipped Tool 11.4 (set RUN_TOOL = True and ACTION='upload'/'download').")

### 工具 11.5：清理临时残片与原始 Parquet（释放磁盘空间）

In [ ]:
RUN_TOOL = False          # 单独运行时改成 True
REMOVE_PARQUETS = False   # 确认所有转换已完成后，设为 True 可释放 /kaggle/tmp 下 ~451 MB 原始 Parquet

if RUN_TOOL:
    from pathlib import Path
    from mahjong_rl.data.hf_sync import clean_tmp_and_cache, inspect_kaggle_resources

    _work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    _tmp = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else (_work / "tmp")
    _shard_dir = globals().get("SHARD_DIR", _work / "v6_shards_2026")
    _parquet_dir = globals().get("PARQUET_DIR", _tmp / "tenhou_2026_mjai")
    clean_tmp_and_cache(_shard_dir, _parquet_dir, remove_parquets=REMOVE_PARQUETS)
    inspect_kaggle_resources(_work, _tmp)
else:
    print("Skipped Tool 11.5 (set RUN_TOOL = True to clean tmp files / parquets).")

### 工具 11.6：热更新仓库代码并清空 `mahjong_rl` 内存缓存（免重启内核拉取最新修复）

In [ ]:
RUN_TOOL = False  # 单独运行时改成 True

if RUN_TOOL:
    import subprocess, sys
    from pathlib import Path

    _work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    _repo = globals().get("REPO_DIR", _work / "mahjax")
    _branch = globals().get("REPO_BRANCH", "main")
    subprocess.run(["git", "-C", str(_repo), "fetch", "origin", _branch], check=False)
    subprocess.run(
        ["git", "-C", str(_repo), "checkout", "-B", _branch, f"origin/{_branch}"], check=False
    )
    evicted = [m for m in list(sys.modules) if m == "mahjong_rl" or m.startswith("mahjong_rl.")]
    for _m in evicted:
        del sys.modules[_m]
    head = subprocess.check_output(["git", "-C", str(_repo), "log", "-1", "--oneline"], text=True).strip()
    branch = subprocess.check_output(
        ["git", "-C", str(_repo), "rev-parse", "--abbrev-ref", "HEAD"], text=True
    ).strip()
    print(f"[Hot Reload OK] branch={branch} HEAD={head} | evicted {len(evicted)} cached mahjong_rl modules")
else:
    print("Skipped Tool 11.6 (set RUN_TOOL = True to git pull & reload mahjong_rl without restart).")

### 工具 11.7：离线批量回测历史版本 / `outer_snapshots` 对战曲线（立即可用于补测当前已跑模型）

> 单独运行时请先确保已执行第 **1**、**2**、**7.5** 节（加载 `CompiledDuplicate1v3Evaluator`），然后将下方 `RUN_TOOL = True`。

In [ ]:
RUN_TOOL = False                 # 单独运行时改成 True
BACKTEST_WALLS = 128             # 每组回测牌山数（128 = 512 场半庄，95% CI ~ ±6.8 pt）
BACKTEST_SEED = 2026             # 验证牌山种子（所有候选在同一牌山上 4 席配对）
BACKTEST_CHUNK_WALLS = 8
INCLUDE_OUTER_SNAPSHOTS = True   # True=一并回测 rl_resume_state.pkl 中的历史 outer_snapshots（配对最新 B_EMA）
PROMOTE_BEST_TO_RL_BEST = False  # True=把本次实测最高分的候选晋升保存为 rl_best.pkl

if RUN_TOOL:
    import json
    import pickle
    import re
    from pathlib import Path
    import jax
    import matplotlib.pyplot as plt
    import numpy as np

    if "CompiledDuplicate1v3Evaluator" not in globals():
        raise RuntimeError("请先运行上方第 7.5 节单元格以加载 CompiledDuplicate1v3Evaluator")

    from mahjong_rl.algo.utility import W_PHOENIX_7D
    from mahjong_rl.config import BeliefConfig, EvalConfig, GRPConfig, ModelConfig
    from mahjong_rl.models.belief import BeliefNet
    from mahjong_rl.models.equivariant import SovereignPolicyNet
    from mahjong_rl.models.grp import GRPNet

    _work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
    _ckpt_dir = globals().get("CKPT_DIR", _work / "checkpoints" / "sovereign_v6")

    def _load_pkl(p: Path):
        with open(p, "rb") as f:
            return pickle.load(f)

    d_grp = _load_pkl(_ckpt_dir / "grp_phase0b.pkl")
    d_bel = _load_pkl(_ckpt_dir / "belief_phase0a.pkl")
    d_bc = _load_pkl(_ckpt_dir / "bc_phase0c.pkl")

    bt_grp_cfg = GRPConfig(**d_grp["config"])
    bt_grp_model = GRPNet(config=bt_grp_cfg)
    bt_grp_params = jax.device_get(d_grp["params"])

    bt_bel_cfg = BeliefConfig(**d_bel["config"])
    bt_bel_model = BeliefNet(config=bt_bel_cfg)
    bt_bel_params = jax.device_get(d_bel["params"])
    bt_t_cal = float(d_bc.get("t_cal", d_bel.get("t_cal", 1.0)))

    bt_model_cfg = ModelConfig(**d_bc["model_config"])
    bt_pol_model = SovereignPolicyNet(config=bt_model_cfg)
    bt_bc_params = jax.device_get(d_bc["params"])

    # Discover all checkpoints in CKPT_DIR
    # Tuple: (label, step_hint, pol_params, bel_params, note)
    bt_entries = []

    def _upd_num(path: Path) -> int:
        m = re.search(r"rl_upd_(\d+)\.pkl$", path.name)
        return int(m.group(1)) if m else -1

    for p in sorted(_ckpt_dir.glob("rl_upd_*.pkl"), key=_upd_num):
        d = _load_pkl(p)
        bt_entries.append((
            p.stem,
            int(d.get("completed_updates", _upd_num(p))),
            jax.device_get(d["policy_params"]),
            jax.device_get(d["belief_ema_params"]),
            p.name,
        ))

    if (_ckpt_dir / "rl_best.pkl").exists():
        d = _load_pkl(_ckpt_dir / "rl_best.pkl")
        bt_entries.append((
            f"rl_best_u{int(d.get('completed_updates', 0)):04d}",
            int(d.get("completed_updates", 0)),
            jax.device_get(d["policy_params"]),
            jax.device_get(d["belief_ema_params"]),
            "rl_best.pkl",
        ))

    if (_ckpt_dir / "rl_phase2.pkl").exists():
        d = _load_pkl(_ckpt_dir / "rl_phase2.pkl")
        u = int(d.get("completed_updates", len(d.get("history") or [])))
        bt_entries.append((
            f"rl_phase2_u{u:04d}",
            u,
            jax.device_get(d.get("policy_params", d.get("params_pi"))),
            jax.device_get(d.get("belief_ema_params", d.get("params_b_ema"))),
            "rl_phase2.pkl",
        ))

    if (_ckpt_dir / "deploy_sdtap.pkl").exists():
        d = _load_pkl(_ckpt_dir / "deploy_sdtap.pkl")
        bt_entries.append((
            "deploy_sdtap",
            999999,
            jax.device_get(d["policy_params"]),
            jax.device_get(d["belief_ema_params"]),
            "deploy_sdtap.pkl",
        ))

    if (_ckpt_dir / "rl_resume_state.pkl").exists():
        d_res = _load_pkl(_ckpt_dir / "rl_resume_state.pkl")
        u_cur = int(d_res.get("completed_updates", len(d_res.get("history") or [])))
        bel_cur = jax.device_get(d_res.get("params_b_ema", bt_bel_params))
        if INCLUDE_OUTER_SNAPSHOTS and d_res.get("outer_snapshots"):
            snaps = d_res["outer_snapshots"]
            for i, sp in enumerate(snaps):
                bt_entries.append((
                    f"outer_snap_{i}",
                    i,
                    jax.device_get(sp),
                    bel_cur,
                    f"rl_resume_state.outer_snapshots[{i}/{len(snaps)-1}] (paired w/ latest B_EMA)",
                ))
        bt_entries.append((
            f"resume_cur_u{u_cur:04d}",
            u_cur,
            jax.device_get(d_res.get("params_pi", d_res.get("policy_params"))),
            bel_cur,
            "rl_resume_state.pkl (current)",
        ))
        del d_res

    if not bt_entries:
        print(f"[Backtest] No RL checkpoints found in {_ckpt_dir}")
    else:
        bt_eval_cfg = EvalConfig(
            env_name="red_4p",
            round_mode="half",
            num_walls=BACKTEST_WALLS,
            max_steps=2400,
            rank_points=(90.0, 45.0, 0.0, -135.0),
            seed=BACKTEST_SEED,
        )
        bt_eval = CompiledDuplicate1v3Evaluator(
            policy_model=bt_pol_model,
            belief_model=bt_bel_model,
            grp_model=bt_grp_model,
            grp_base_params=bt_grp_params,
            eval_cfg=bt_eval_cfg,
            w_rule=W_PHOENIX_7D,
            opp_rules_known=False,
            belief_t_cal=bt_t_cal,
            greedy=True,
            chunk_walls=BACKTEST_CHUNK_WALLS,
        )

        print(f"=== Offline Backtest vs 3x phase0c_bc_7d ({4 * BACKTEST_WALLS} games/entry | seed={BACKTEST_SEED}) ===")
        bt_results = []
        for label, step_hint, pol_p, bel_p, note in bt_entries:
            res = bt_eval.evaluate(pol_p, bel_p, bt_bc_params, bt_bel_params)
            bt_results.append({
                "label": label,
                "step_hint": step_hint,
                "note": note,
                "res": res,
                "_pol": pol_p,
                "_bel": bel_p,
            })
            print(
                f"  [{label:22s}] pt={res['tenhou_pt_mean']:+6.2f}±{res['tenhou_pt_ci95']:5.2f} | "
                f"rank={res['avg_rank']:.3f} | 1st={res['rank_1_rate']*100:4.1f}% 4th={res['rank_4_rate']*100:4.1f}% | "
                f"hora={res['hora_rate']*100:4.1f}% deal={res['deal_in_rate']*100:4.1f}% | "
                f"{res['eval_wall_s']:.1f}s | {note}"
            )

        ranked = sorted(bt_results, key=lambda x: x["res"]["tenhou_pt_mean"], reverse=True)
        print("\n=== Ranked Leaderboard (Best -> Worst vs 3x BC) ===")
        for rank_i, item in enumerate(ranked, 1):
            r = item["res"]
            print(
                f"  #{rank_i:<2d} {item['label']:22s} | pt={r['tenhou_pt_mean']:+6.2f}±{r['tenhou_pt_ci95']:5.2f} | "
                f"rank={r['avg_rank']:.3f} | 1st={r['rank_1_rate']*100:4.1f}% 4th={r['rank_4_rate']*100:4.1f}%"
            )

        top = ranked[0]
        if PROMOTE_BEST_TO_RL_BEST:
            best_payload = {
                "completed_updates": int(top["step_hint"]),
                "model_config": d_bc["model_config"],
                "belief_config": d_bel["config"],
                "grp_config": d_grp["config"],
                "policy_params": top["_pol"],
                "belief_ema_params": top["_bel"],
                "grp_base_params": bt_grp_params,
                "t_cal": float(bt_t_cal),
                "eval_vs_bc": top["res"],
                "eval_vs_best": None,
                "promoted_from": top["label"],
            }
            _atomic_dump_pkl(_ckpt_dir / "rl_best.pkl", best_payload)
            print(f"\n[Promoted] Saved {top['label']} -> {_ckpt_dir / 'rl_best.pkl'} (pt={top['res']['tenhou_pt_mean']:+.2f})")

        # Save JSON & plot comparison chart
        bt_json = [
            {"label": x["label"], "step_hint": x["step_hint"], "note": x["note"], **x["res"]}
            for x in bt_results
        ]
        _atomic_write_json(_ckpt_dir / "backtest_report.json", bt_json)

        fig, ax = plt.subplots(figsize=(11, max(3.5, 0.48 * len(bt_results) + 1.5)))
        y_pos = np.arange(len(bt_results))
        means = [x["res"]["tenhou_pt_mean"] for x in bt_results]
        errs = [x["res"]["tenhou_pt_ci95"] for x in bt_results]
        labels = [x["label"] for x in bt_results]
        colors = ["#d62728" if x["label"] == top["label"] else "#1f77b4" for x in bt_results]
        ax.barh(y_pos, means, xerr=errs, color=colors, alpha=0.85, capsize=4)
        ax.axvline(0.0, color="#555555", linestyle="--", linewidth=1)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_xlabel("Paired Tenhou pt vs 3x phase0c_bc_7d (±95% CI)")
        ax.set_title(f"Offline Checkpoint Backtest ({4 * BACKTEST_WALLS} games/entry, seed={BACKTEST_SEED})")
        ax.grid(True, axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig(_ckpt_dir / "backtest_curves.png", dpi=140)
        plt.show()
        plt.close(fig)
else:
    print("Skipped Tool 11.7 (run Section 7.5 first, then set RUN_TOOL = True to backtest all saved versions / outer_snapshots).")